# 🐟 Tarea 2 — Parte 1: CNN propia y YOLO desde cero

## Objetivo

Implementar y comparar **dos detectores de objetos entrenados desde cero** con imágenes submarinas del dataset **FjordFish**: (1) una CNN con arquitectura propia y (2) una arquitectura YOLO. Cada detector debe localizar organismos mediante *bounding boxes* e identificar su clase.

## Qué incluye este cuaderno

- **Dataset:** descarga de `FjordFish.zip` desde Zenodo **directamente a tu Google Drive**, verificación de integridad y extracción temporal en Colab. También admite un ZIP ya guardado o una carpeta de datos extraídos.
- **Preparación de datos:** auditoría de imágenes y etiquetas YOLO, manejo de incidencias documentadas y particiones comunes `train/val/test`.
- **Modelos:** CNN + SSD y YOLO desde cero, sin cargar pesos preentrenados.
- **Ejecución:** pruebas técnicas, entrenamiento, validación, guardado de checkpoints y evaluación en el conjunto de prueba.
- **Salidas:** precisión, recall, AP50, mAP50–95, tiempos, curvas, archivos CSV/JSON y ejemplos con cajas detectadas.

## Cómo ejecutarlo en Google Colab

1. Selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** (o una GPU compatible).
2. Comprueba que tu Google Drive tenga **más de 6,1 GB libres** para guardar el ZIP. Colab necesita espacio temporal adicional para descomprimirlo.
3. Selecciona **Entorno de ejecución → Ejecutar todas** y autoriza la conexión con Google Drive.
4. El notebook descargará `FjordFish.zip` desde Zenodo a `Mi unidad/Tarea2_DL/data/` si aún no existe, verificará su MD5 y utilizará la copia guardada en las ejecuciones siguientes.
5. Los checkpoints, registros y gráficos se guardan bajo `Mi unidad/Tarea2_DL/resultados_parte_1/`.

> **Importante:** No necesitas descargar el dataset en Ubuntu ni subir el ZIP manualmente. Una descarga de 6,1 GB puede fallar por cuota de Drive, cortes de conexión o límites de Colab. Si se interrumpe, vuelve a ejecutar: el descargador guarda un archivo `.part` para intentar continuar.

> **Alcance académico:** este notebook contiene la infraestructura y las implementaciones existentes, **no resultados reales previamente ejecutados**. Las decisiones y justificaciones de diseño y entrenamiento, y la interpretación de resultados, corresponden al grupo. El enunciado restringe expresamente el uso de IA para diseñar soluciones o analizar resultados; se debe declarar la asistencia recibida y confirmar su admisibilidad con el docente.


## 1. Configuración técnica

Se conservan las arquitecturas y los parámetros que estaban en el código aportado; las cantidades de épocas se dejan con **valores provisionales de arranque** para evitar errores por `None`. El grupo debe validarlas y justificar su propia elección antes de la entrega.


In [ ]:
from pathlib import Path

# Ejecutar en Google Colab con GPU (Entorno de ejecución > Cambiar tipo > T4 GPU).
# El ZIP se descargará desde Zenodo a Drive sin pasar por tu Ubuntu.
MODO_SINTETICO = False  # True: diagnóstico opcional sin peces, NO sirve como resultado de tarea.
MONTAR_DRIVE = True
EXIGIR_GPU = True
DESCARGAR_DATASET = True   # Descarga FjordFish.zip desde Zenodo directamente a Drive si falta.
ZIP_LOCAL = None            # Opcional: ruta de un ZIP ya descargado, si lo tienes.
DATASET_EXISTENTE = None    # Si ya descomprimiste FjordFish, coloca ruta opcional.
YAML_CATALOGO = None
GRUPOS_CSV = None

PROYECTO_DRIVE = Path('/content/drive/MyDrive/Tarea2_DL')
TRABAJO_LOCAL = Path('/content/Tarea2_DL')
RUN_ID = 'parte1_provisional_10ep_2026'  # Cambiar para experimentos distintos.
SEED = 42
TAMANO = 512
PROPORCIONES = (0.70, 0.15, 0.15)
DISTANCIA_DHASH = 4
APLICAR_CURACION = True
REUTILIZAR_AUDITORIA = True
REANUDAR = True
PRUEBA_INTEGRACION = True
METRICA_SELECCION = 'map_50_95'
BATCH_EVALUACION = 2

# ¡CONFIGURACIÓN PROVISIONAL SOLO PARA PODER EJECUTAR EL FLUJO!
# 10 épocas por detector NO son una decisión justificada del grupo ni garantizan calidad.
# El grupo debe decidir/justificar los hiperparámetros y las épocas para la entrega.
EPOCAS_CNN = 10
EPOCAS_YOLO = 10
CONFIG_MODELOS = {
    'cnn_ssd': {'epochs': EPOCAS_CNN, 'batch_size': 2, 'learning_rate': 0.001,
                'weight_decay': 0.0001, 'augment': False},
    'yolo_scratch': {'epochs': EPOCAS_YOLO, 'batch_size': 2, 'learning_rate': 0.001,
                     'weight_decay': 0.0001, 'augment': False},
}


## 2. Rutas y almacenamiento

Colab solicitará que autorices el montaje de Drive. En modo sintético se trabaja en disco temporal y no se monta Drive.


In [ ]:
import os, sys, shutil, json

if MODO_SINTETICO:
    PROYECTO = TRABAJO_LOCAL / "demo_sintetica"
else:
    if MONTAR_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        if not Path("/content/drive/MyDrive").is_dir():
            raise RuntimeError("Drive no quedó montado. Completa la autorización antes de continuar.")
    if str(PROYECTO_DRIVE).startswith("/content/drive/") and not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("La ruta elegida está en Drive pero Drive no está montado. Monta Drive o configura una carpeta local.")
    PROYECTO = PROYECTO_DRIVE

PROYECTO.mkdir(parents=True, exist_ok=True)
TRABAJO_LOCAL.mkdir(parents=True, exist_ok=True)
os.chdir(TRABAJO_LOCAL)
DATA_DIR = PROYECTO / "data"
DATASET_DIR = Path(DATASET_EXISTENTE).expanduser().resolve() if DATASET_EXISTENTE else TRABAJO_LOCAL / "data/extracted"
RESULTADOS = PROYECTO / "resultados_parte_1" / RUN_ID
CODIGO = TRABAJO_LOCAL / "codigo_parte1"
RESULTADOS.mkdir(parents=True, exist_ok=True)
CODIGO.mkdir(parents=True, exist_ok=True)
os.environ["YOLO_CONFIG_DIR"] = str(TRABAJO_LOCAL / "ultralytics")
os.environ["YOLO_AUTOINSTALL"] = "false"
print("ZIP/descarga persistente:", DATA_DIR)
print("Datos extraídos:", DATASET_DIR)
print("Resultados y checkpoints:", RESULTADOS)
print("Espacio libre en trabajo local (GiB):", round(shutil.disk_usage(TRABAJO_LOCAL).free / 1024**3, 2))
print("Espacio informado en proyecto (GiB):", round(shutil.disk_usage(PROYECTO).free / 1024**3, 2))


Mounted at /content/drive
ZIP/descarga persistente: /content/drive/MyDrive/Tarea2_DL/data
Datos extraídos: /content/Tarea2_DL/data/extracted
Resultados y checkpoints: /content/drive/MyDrive/Tarea2_DL/resultados_parte_1/parte1_provisional_10ep_2026
Espacio libre en trabajo local (GiB): 70.06
Espacio informado en proyecto (GiB): 66.55


## 3. Dependencias

Se fija Ultralytics en la versión del cuaderno original y se usa la pareja PyTorch/Torchvision del entorno Colab. Si esas dos librerías no importan correctamente, se detiene antes de descargar datos.


In [ ]:
import importlib.util, importlib.metadata, subprocess

try:
    import torch, torchvision
except Exception as error:
    raise RuntimeError("Usa un entorno Colab con torch/torchvision compatibles antes de continuar") from error

requisitos = {
    "ultralytics": "ultralytics==8.4.174", "numpy": "numpy>=1.26", "PIL": "Pillow>=10",
    "yaml": "PyYAML>=6", "matplotlib": "matplotlib>=3.8", "pycocotools": "pycocotools>=2.0.8",
}
instalar = [spec for module, spec in requisitos.items() if importlib.util.find_spec(module) is None]
try:
    version_yolo = importlib.metadata.version("ultralytics")
except importlib.metadata.PackageNotFoundError:
    version_yolo = None
if version_yolo not in (None, "8.4.174"):
    if "ultralytics" in sys.modules:
        raise RuntimeError("Ya se importó otra versión de Ultralytics. Reinicia la sesión y ejecuta desde el inicio.")
    instalar.append("ultralytics==8.4.174")
if instalar:
    subprocess.run([sys.executable, "-m", "pip", "install", *instalar], check=True)
import ultralytics
assert ultralytics.__version__ == "8.4.174", "Reinicia el kernel tras instalar la versión fijada"
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__, "| ultralytics:", ultralytics.__version__)


WARNING ⚠️ user config directory '/content/Tarea2_DL/ultralytics/Ultralytics' is not writable, using '/tmp/Ultralytics'. Set YOLO_CONFIG_DIR to override.
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
torch: 2.11.0+cu130 | torchvision: 0.26.0+cu130 | ultralytics: 8.4.174


## 4. Código de los modelos y funciones auxiliares

Los cuatro módulos siguientes están **escritos en Python normal**, en celdas ejecutables. Google Colab los guarda automáticamente como archivos temporales con `%%writefile` para poder importarlos. No debes crear ni subir archivos adicionales. Los códigos corresponden a los scripts existentes, sin sustituir las arquitecturas.

Puedes dejar estas celdas plegadas si solamente quieres ejecutar la tarea.


### 4.1 CNN + SSD, YOLO y utilidades — `base_parte1.py` (11 bloques)


In [ ]:
%%writefile /content/Tarea2_DL/codigo_parte1/base_parte1.py
"""Funciones del cuaderno original; modelos de la parte 1, sin pesos COCO."""

SEED = 42
KINDS = ("cnn_ssd", "yolo_scratch")

# Librerías de archivos, imágenes, cálculo numérico y modelos.
import copy
import csv
import hashlib
import importlib.metadata
import io
import json
import numpy as np
import os
import platform
import random
import re
import shutil
import sys
import tempfile
import time
import torch
import urllib.request
import yaml
import zipfile
from PIL import Image, ImageOps, ImageDraw
from collections import Counter, defaultdict, OrderedDict
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import redirect_stdout
from dataclasses import dataclass
from pathlib import Path
from torch import nn
from torch.utils.data import Dataset
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.anchor_utils import DefaultBoxGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.image_list import ImageList
from torchvision.models.detection.ssd import SSD
from torchvision.ops import FrozenBatchNorm2d


# Lee el informe guardado; mantener JSON facilita revisar los resultados fuera de Python.
def leer_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


# Escribe primero un temporal para evitar un JSON incompleto si se interrumpe el proceso.
def guardar_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


# El hash verifica el contenido; un nombre de archivo igual no garantiza pesos iguales.
def calcular_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# YOLO usa centro y tamaño normalizados; el evaluador trabaja con coordenadas en píxeles.
def yolo_a_cajas(box, width, height):
    cx, cy, w, h = map(float, box)
    return [
        (cx - w / 2) * width,
        (cy - h / 2) * height,
        (cx + w / 2) * width,
        (cy + h / 2) * height,
    ]


# La conversión inversa permite exportar las anotaciones depuradas sin releer etiquetas erróneas.
def cajas_a_yolo(box, width, height):
    x1, y1, x2, y2 = map(float, box)
    return [
        (x1 + x2) / (2 * width),
        (y1 + y2) / (2 * height),
        (x2 - x1) / width,
        (y2 - y1) / height,
    ]


# IoU mide intersección sobre unión; el pequeño denominador evita divisiones por cero.
def iou_cajas(a, b):
    a = np.asarray(a, dtype=float).reshape(-1, 4)
    b = np.asarray(b, dtype=float).reshape(-1, 4)
    lo = np.maximum(a[:, None, :2], b[None, :, :2])
    hi = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.maximum(hi - lo, 0)
    inter = wh[..., 0] * wh[..., 1]
    area_a = np.prod(np.maximum(a[:, 2:] - a[:, :2], 0), axis=1)
    area_b = np.prod(np.maximum(b[:, 2:] - b[:, :2], 0), axis=1)
    return inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-12)

# Guarda escala y márgenes para devolver cada caja al tamaño original antes de evaluar.
@dataclass(frozen=True)
class AjusteImagen:
    width: int
    height: int
    size: int
    resized_width: int
    resized_height: int
    left: int
    top: int

    def forward_boxes(self, boxes):
        result = np.asarray(boxes, dtype=np.float32).reshape(-1, 4).copy()
        result *= [
            self.resized_width / self.width,
            self.resized_height / self.height,
        ] * 2
        result += [self.left, self.top] * 2
        return result

    def inverse_boxes(self, boxes):
        result = np.asarray(boxes, dtype=np.float32).reshape(-1, 4).copy()
        result -= [self.left, self.top] * 2
        result /= [
            self.resized_width / self.width,
            self.resized_height / self.height,
        ] * 2
        result[:, [0, 2]] = result[:, [0, 2]].clip(0, self.width)
        result[:, [1, 3]] = result[:, [1, 3]].clip(0, self.height)
        return result


# Letterbox conserva la proporción y completa con margen; evita deformar los animales.
def ajustar_imagen(image, size=512):
    width, height = image.size
    scale = min(size / width, size / height)
    rw, rh = (max(1, round(width * scale)), max(1, round(height * scale)))
    left, top = ((size - rw) // 2, (size - rh) // 2)
    result = Image.new("RGB", (size, size), (114, 114, 114))
    result.paste(
        image.convert("RGB").resize((rw, rh), Image.Resampling.BILINEAR), (left, top)
    )
    return (result, AjusteImagen(width, height, size, rw, rh, left, top))


# Si se refleja una imagen, sus cajas deben reflejarse con la misma transformación.


Writing /content/Tarea2_DL/codigo_parte1/base_parte1.py


In [ ]:
%%writefile -a /content/Tarea2_DL/codigo_parte1/base_parte1.py
def reflejar_imagen(image, boxes):
    result = np.asarray(boxes, dtype=np.float32).reshape(-1, 4).copy()
    result[:, [0, 2]] = image.width - result[:, [2, 0]]
    return (ImageOps.mirror(image), result)

# NMS se aplica por clase: dos especies distintas no se eliminan por compartir una zona.
def filtrar_detecciones(
    boxes, labels, scores, score_min=0.001, nms_iou=0.5, max_detections=100
):
    """NMS común por clase, estable frente a empates y sin dependencia de torch."""
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    labels = np.asarray(labels, dtype=int).reshape(-1)
    scores = np.asarray(scores, dtype=float).reshape(-1)
    if not len(boxes) == len(labels) == len(scores):
        raise ValueError("Cajas, clases y scores deben tener igual longitud")
    valid = np.isfinite(boxes).all(axis=1) & np.isfinite(scores)
    valid &= (boxes[:, 2:] > boxes[:, :2]).all(axis=1) & (scores >= score_min)
    indices = np.flatnonzero(valid)
    order = indices[np.argsort(-scores[indices], kind="stable")]
    keep = []
    while len(order) and len(keep) < max_detections:
        current, rest = (order[0], order[1:])
        keep.append(current)
        overlaps = iou_cajas(boxes[current : current + 1], boxes[rest])[0]
        order = rest[(labels[rest] != labels[current]) | (overlaps <= nms_iou)]
    return (boxes[keep], labels[keep], scores[keep])

# Formatos de imagen reconocidos en la auditoría.
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# La firma de rutas, tamaños y fechas invalida la caché si cambia la fuente; no sustituye los hashes de contenido.
def firma_archivos(root):
    import json

    root = Path(root)
    entries = []
    for p in sorted(root.rglob("*")):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS | {".txt", ".yaml", ".yml"}:
            stat = p.stat()
            entries.append((p.relative_to(root).as_posix(), stat.st_size, stat.st_mtime_ns))
    return hashlib.sha256(json.dumps(entries).encode()).hexdigest()


# El YAML define los identificadores; conservarlos evita cambiar el significado de las clases.
def leer_clases(path):
    config = yaml.safe_load(Path(path).read_text(encoding="utf-8-sig"))
    names = config.get("names")
    if isinstance(names, list):
        names = dict(enumerate(names))
    if not isinstance(names, dict) or not names:
        raise ValueError("El YAML debe declarar names como lista o diccionario")
    names = {int(k): re.sub("^\\d+\\s*:\\s*", "", str(v)).strip() for k, v in names.items()}
    if sorted(names) != list(range(len(names))):
        raise ValueError("Los IDs del catálogo deben ser contiguos desde cero")
    if len(set(names.values())) != len(names):
        raise ValueError("Nombres de clases duplicados")
    return [names[i] for i in range(len(names))]


# La carpeta labels debe corresponder a la ruta de cada imagen, no solo a su nombre.
def ruta_etiqueta(image_path):
    parts = list(image_path.parts)
    matches = [i for i, part in enumerate(parts) if part.lower() == "images"]
    if not matches:
        raise ValueError(f"La imagen no está bajo un directorio images: {image_path}")
    parts[matches[-1]] = "labels"
    return Path(*parts).with_suffix(".txt")


# Una etiqueta ausente es un error; un archivo vacío sí representa una imagen sin objetos.
def leer_etiquetas(path, width, height, num_classes, allow_missing=False):
    if not path.exists():
        if allow_missing:
            return []
        raise ValueError(f"Etiqueta ausente: {path}")
    result = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), 1):
        if not line.strip():
            continue
        tokens = line.split()
        if len(tokens) != 5:
            raise ValueError(f"{path}:{line_number}: se esperan class cx cy w h")
        values = np.asarray([float(x) for x in tokens], dtype=float)
        category = int(values[0]) if np.isfinite(values[0]) else -1
        if not np.isfinite(values).all() or category != values[0] or (not 0 <= category < num_classes):
            raise ValueError(f"{path}:{line_number}: clase o valores inválidos")
        cx, cy, w, h = values[1:]
        if not (0 <= cx <= 1 and 0 <= cy <= 1 and (0 < w <= 1) and (0 < h <= 1)):
            raise ValueError(f"{path}:{line_number}: coordenadas YOLO inválidas")
        box = yolo_a_cajas(values[1:], width, height)
        tolerance = 0.0001 * max(width, height)
        if min(box) < -tolerance or box[2] > width + tolerance or box[3] > height + tolerance:
            raise ValueError(f"{path}:{line_number}: caja fuera de la imagen")
        box = [max(0.0, box[0]), max(0.0, box[1]), min(float(width), box[2]), min(float(height), box[3])]
        result.append({"category_id": category, "bbox_xyxy": box})
    return result


def inspeccionar_imagenes(paths, root, names, allow_missing):
    """Lee dimensiones, hashes y cajas; informa los archivos que no pasan la revisión."""
    rows, errors = [], []
    used_labels = set()
    for path in paths:
        try:
            # No corregir EXIF automáticamente: podría desalinear las cajas.
            with Image.open(path) as raw:
                if raw.getexif().get(274, 1) != 1:
                    raise ValueError(
                        f"{path}: orientación EXIF distinta de 1; revisar coordenadas antes de corregir"
                    )
                image = raw.convert("RGB")
                width, height = image.size
                small = np.asarray(image.convert("L").resize((9, 8)))
                bits = (small[:, 1:] > small[:, :-1]).reshape(-1)
                dhash = int("".join(("1" if b else "0" for b in bits)), 2)
                pixel_hash = hashlib.sha256(f"{width}x{height}".encode() + image.tobytes()).hexdigest()
            # Separar el hash de píxeles de la huella aproximada dHash.
            labels_file = ruta_etiqueta(path)
            annotations = leer_etiquetas(labels_file, width, height, len(names), allow_missing)
            used_labels.add(labels_file.resolve())
            rows.append(
                {
                    "id": len(rows) + 1,
                    "file_name": path.relative_to(root).as_posix(),
                    "width": width,
                    "height": height,
                    "sha256": calcular_sha256(path),
                    "pixel_sha256": pixel_hash,
                    "dhash": f"{dhash:016x}",
                    "annotations": annotations,
                }
            )
        except (ValueError, OSError) as exc:
            errors.append(str(exc))
    return rows, errors, used_labels


# La unión es transitiva: si A se parece a B y B a C, se conserva el grupo A-B-C.
class GruposConectados:

    def __init__(self, size):
        self.parent = list(range(size))

    def find(self, item):
        while self.parent[item] != item:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, a, b):
        self.parent[self.find(b)] = self.find(a)




Appending to /content/Tarea2_DL/codigo_parte1/base_parte1.py


In [ ]:
%%writefile -a /content/Tarea2_DL/codigo_parte1/base_parte1.py
def agrupar_similares(rows, groups_csv, near_distance, errors, warnings):
    """Une imágenes relacionadas para impedir que un grupo se reparta entre conjuntos."""
    uf = GruposConectados(len(rows))
    exact, near = ([], [])
    seen_hash, seen_dhash = ({}, {})
    for i, row in enumerate(rows):
        sha = row["pixel_sha256"]
        # Mismos píxeles con etiquetas distintas requieren revisión, no una elección arbitraria.
        if sha in seen_hash:
            other = seen_hash[sha]
            exact.append([rows[other]["id"], row["id"]])
            uf.union(i, other)
            a = sorted(((x["category_id"], tuple(np.round(x["bbox_xyxy"], 3))) for x in row["annotations"]))
            b = sorted(
                ((x["category_id"], tuple(np.round(x["bbox_xyxy"], 3))) for x in rows[other]["annotations"])
            )
            if a != b:
                errors.append(
                    f"Duplicados con etiquetas diferentes: {row['file_name']} y {rows[other]['file_name']}"
                )
        seen_hash[sha] = i
        dh = int(row["dhash"], 16)
        # dHash detecta similitud visual; no demuestra que sean cuadros de un mismo video.
        if near_distance >= 0:
            for previous, indices in seen_dhash.items():
                if (dh ^ previous).bit_count() <= near_distance:
                    other = indices[0]
                    uf.union(i, other)
                    if rows[other]["pixel_sha256"] != sha:
                        near.append([rows[other]["id"], row["id"]])
            seen_dhash.setdefault(dh, []).append(i)
    # Si existen metadatos de secuencia, sumarlos al agrupamiento visual.
    if groups_csv:
        declared = {}
        covered_paths = set()
        by_path = {r["file_name"]: i for i, r in enumerate(rows)}
        with Path(groups_csv).open(encoding="utf-8-sig", newline="") as stream:
            for row in csv.DictReader(stream):
                key = row["file_name"].replace("\\", "/")
                if key not in by_path or not row.get("group_id"):
                    raise ValueError(f"Grupo inválido: {row}")
                i = by_path[key]
                covered_paths.add(key)
                if row["group_id"] in declared:
                    uf.union(i, declared[row["group_id"]])
                declared[row["group_id"]] = i
        if len(covered_paths) < len(rows):
            warnings.append(f"Metadatos de secuencia parciales: {len(covered_paths)}/{len(rows)} imágenes")
    else:
        warnings.append("Sin metadatos de video/secuencia: dHash no garantiza independencia entre escenas")
    components = {}
    for i, row in enumerate(rows):
        component = uf.find(i)
        components[component] = min(components.get(component, row["id"]), row["id"])
    for i, row in enumerate(rows):
        row["group_id"] = f"g{components[uf.find(i)]:06d}"
    return exact, near


def resumir_auditoria(rows, names, exact, near, warnings):
    """Cuenta imágenes, objetos, clases y tamaños de cajas sin modificar las anotaciones."""
    counts = Counter((a["category_id"] for r in rows for a in r["annotations"]))
    image_counts = Counter((c for r in rows for c in {a["category_id"] for a in r["annotations"]}))
    group_sizes = Counter((r["group_id"] for r in rows))
    if group_sizes and max(group_sizes.values()) > len(rows) * 0.2:
        warnings.append(
            "Un grupo supera el 20% de las imágenes; revisar pares similares antes de fijar particiones"
        )
    relative_areas = [
        (a["bbox_xyxy"][2] - a["bbox_xyxy"][0])
        * (a["bbox_xyxy"][3] - a["bbox_xyxy"][1])
        / (r["width"] * r["height"])
        for r in rows
        for a in r["annotations"]
    ]
    return {
        "images": len(rows),
        "objects": sum(counts.values()),
        "classes_declared": len(names),
        "classes_observed": len(counts),
        "empty_images": sum((not r["annotations"] for r in rows)),
        "groups": len(group_sizes),
        "exact_duplicate_pairs": len(exact),
        "near_duplicate_pairs": len(near),
        "box_relative_area_quantiles": (
            np.quantile(relative_areas, [0, 0.25, 0.5, 0.75, 1]).tolist() if relative_areas else []
        ),
        "class_counts": [
            {"id": i, "name": n, "objects": counts[i], "images": image_counts[i]}
            for i, n in enumerate(names)
        ],
    }


# Superponer las cajas ayuda a detectar errores visibles; una muestra no valida todo el dataset.
def mostrar_muestra(report, destination, count=12):
    rows = report["images"]
    if not rows:
        return
    indices = np.linspace(0, len(rows) - 1, min(count, len(rows)), dtype=int)
    canvas = Image.new("RGB", (4 * 320, (len(indices) + 3) // 4 * 260), "#202020")
    for pos, index in enumerate(indices):
        row = rows[index]
        with Image.open(Path(report["root"]) / row["file_name"]) as raw:
            image = raw.convert("RGB")
        thumb = ImageOps.contain(image, (320, 230))
        draw = ImageDraw.Draw(thumb)
        sx, sy = (thumb.width / image.width, thumb.height / image.height)
        for ann in row["annotations"]:
            scaled = [v * scale for v, scale in zip(ann["bbox_xyxy"], [sx, sy, sx, sy])]
            draw.rectangle(scaled, outline="yellow", width=2)
            cat = report["categories"][ann["category_id"]]
            draw.text(
                tuple(scaled[:2]),
                f"{cat['id']}: {cat['name']}",
                fill="yellow",
                stroke_width=1,
                stroke_fill="black",
            )
        x, y = (pos % 4 * 320, pos // 4 * 260)
        canvas.paste(thumb, (x, y))
        ImageDraw.Draw(canvas).text((x + 4, y + 235), row["file_name"][-45:], fill="white")
    canvas.save(destination)


# Las distribuciones permiten revisar desbalance, resoluciones y tamaños antes de entrenar.
def graficar_datos(report, output):
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    rows = report["images"]
    if not rows:
        return
    classes = report["summary"]["class_counts"]
    fig, ax = plt.subplots(figsize=(12, max(5, len(classes) * 0.3)))
    y = np.arange(len(classes))
    ax.barh(y, [r["objects"] for r in classes], color="#247a80")
    ax.set_yticks(y, [f"{r['id']}: {r['name']}" for r in classes])
    ax.invert_yaxis()
    ax.set_xlabel("Objetos anotados")
    ax.set_title("Catálogo declarado y soporte real por clase")
    for i, row in enumerate(classes):
        ax.text(row["objects"], i, f"  {row['objects']}", va="center", fontsize=8)
    ax.margins(x=0.1)
    fig.tight_layout()
    fig.savefig(output / "class_distribution.png", dpi=140)
    plt.close(fig)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    axes[0].scatter([r["width"] for r in rows], [r["height"] for r in rows], alpha=0.25, s=12)
    axes[0].set(xlabel="Ancho (px)", ylabel="Alto (px)", title="Resoluciones originales")
    counts = [len(r["annotations"]) for r in rows]
    axes[1].hist(counts, bins=np.arange(max(counts) + 2) - 0.5, color="#247a80")
    axes[1].set(xlabel="Objetos por imagen", ylabel="Imágenes", title="Multiplicidad de objetos")
    areas = [
        (a["bbox_xyxy"][2] - a["bbox_xyxy"][0])
        * (a["bbox_xyxy"][3] - a["bbox_xyxy"][1])
        / (r["width"] * r["height"])
        for r in rows
        for a in r["annotations"]
    ]
    if areas:
        axes[2].hist(np.log10(np.maximum(areas, 1e-12)), bins=35, color="#247a80")
    axes[2].set(xlabel="log10(área caja / área imagen)", ylabel="Objetos", title="Escala de los objetos")
    fig.tight_layout()
    fig.savefig(output / "data_distributions.png", dpi=140)
    plt.close(fig)




Appending to /content/Tarea2_DL/codigo_parte1/base_parte1.py


In [ ]:
%%writefile -a /content/Tarea2_DL/codigo_parte1/base_parte1.py
def guardar_auditoria(report, output):
    """Exporta el informe, el catálogo, los pares similares y los gráficos de revisión."""
    rows, near = report["images"], report["near_pairs"]
    warnings, errors = report["warnings"], report["errors"]
    output.mkdir(parents=True, exist_ok=True)
    guardar_json(output / "audit.json", report)
    guardar_json(output / "catalog.json", report["categories"])
    with (output / "classes.csv").open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["id", "name", "objects", "images"])
        writer.writeheader()
        writer.writerows(report["summary"]["class_counts"])
    mostrar_muestra(report, output / "annotations.jpg")
    if near:
        by_id = {r["id"]: r for r in rows}
        pair_rows = [by_id[i] for pair in near[:12] for i in pair]
        mostrar_muestra({**report, "images": pair_rows}, output / "similarities.jpg", count=len(pair_rows))
    with (output / "similarities.csv").open("w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(["image_a", "image_b", "hamming_distance"])
        by_id = {r["id"]: r for r in rows}
        for a, b in near:
            writer.writerow(
                [
                    by_id[a]["file_name"],
                    by_id[b]["file_name"],
                    (int(by_id[a]["dhash"], 16) ^ int(by_id[b]["dhash"], 16)).bit_count(),
                ]
            )
    graficar_datos(report, output)
    lines = [
        "# Auditoría de datos",
        "",
        f"Válida: {report['valid']}",
        "",
        "```json",
        __import__("json").dumps(report["summary"], indent=2, ensure_ascii=False),
        "```",
        "",
        "## Advertencias",
    ]
    lines += [f"- {w}" for w in warnings] + ["", "## Errores"] + [f"- {e}" for e in errors]
    (output / "audit.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def auditar(root, output, yaml_path=None, groups_csv=None, allow_missing=False, near_distance=4):
    """Coordina la auditoría; los errores quedan registrados y bloquean las particiones."""
    root, output = (Path(root).resolve(), Path(output))
    if not root.is_dir():
        raise ValueError(f"No existe el dataset: {root}")
    yaml_files = sorted(root.rglob("data.y*ml")) if yaml_path is None else [Path(yaml_path)]
    if len(yaml_files) != 1:
        raise ValueError(f"Se requiere un único data.yaml o --yaml explícito; encontrados: {yaml_files}")
    yaml_path = yaml_files[0]
    names = leer_clases(yaml_path)
    paths = sorted(
        (
            p
            for p in root.rglob("*")
            if p.suffix.lower() in IMAGE_EXTENSIONS and "images" in [s.lower() for s in p.parts]
        )
    )
    if not paths:
        raise ValueError("No se encontraron imágenes bajo images/")
    rows, errors, used_labels = inspeccionar_imagenes(paths, root, names, allow_missing)
    warnings = []
    orphan = [
        p.relative_to(root).as_posix()
        for p in root.rglob("*.txt")
        if "labels" in [s.lower() for s in p.parts] and p.resolve() not in used_labels
    ]
    if orphan:
        errors.append(f"{len(orphan)} etiquetas huérfanas o asociadas a imágenes inválidas")
    exact, near = agrupar_similares(rows, groups_csv, near_distance, errors, warnings)
    summary = resumir_auditoria(rows, names, exact, near, warnings)
    report = {
        "schema_version": 1,
        "root": str(root),
        "yaml": str(yaml_path.resolve()),
        "yaml_sha256": calcular_sha256(yaml_path),
        "categories": [{"id": i, "name": name} for i, name in enumerate(names)],
        "images": rows,
        "errors": errors,
        "warnings": warnings,
        "summary": summary,
        "exact_pairs": exact,
        "near_pairs": near,
        "orphan_labels": orphan,
        "valid": not errors,
        "source_files_signature": firma_archivos(root),
    }
    guardar_auditoria(report, output)
    return report


def recuperar_etiquetas(report, source, root):
    """Reasocia una etiqueta solo si existe una coincidencia huérfana inequívoca."""
    known = {r["file_name"] for r in report["images"]}
    recovered = []
    orphan_paths = {str((root / name).resolve()) for name in source["orphan_labels"]}
    for path in sorted(root.rglob("*")):
        if path.suffix.lower() not in IMAGE_EXTENSIONS or "images" not in [p.lower() for p in path.parts]:
            continue
        if path.relative_to(root).as_posix() in known:
            continue
        expected = ruta_etiqueta(path)
        if expected.exists():
            raise ValueError(f"Imagen inválida por una causa no reparable automáticamente: {path}")
        # Reasociar solo si hay exactamente un archivo huérfano con el mismo identificador.
        candidates = [p for p in root.rglob(path.stem + ".txt") if str(p.resolve()) in orphan_paths]
        if len(candidates) != 1:
            raise ValueError(f"No hay sidecar huérfano único para {path}")
        with Image.open(path) as raw:
            if raw.getexif().get(274, 1) != 1:
                raise ValueError("No se corrige automáticamente orientación EXIF")
            image = raw.convert("RGB")
            width, height = image.size
            small = np.asarray(image.convert("L").resize((9, 8)))
            dhash = int("".join(("1" if b else "0" for b in (small[:, 1:] > small[:, :-1]).reshape(-1))), 2)
            pixel_hash = hashlib.sha256(f"{width}x{height}".encode() + image.tobytes()).hexdigest()
        identifier = max((r["id"] for r in report["images"])) + 1
        row = {
            "id": identifier,
            "file_name": path.relative_to(root).as_posix(),
            "width": width,
            "height": height,
            "sha256": calcular_sha256(path),
            "pixel_sha256": pixel_hash,
            "dhash": f"{dhash:016x}",
            "group_id": f"g{identifier:06d}",
            "annotations": leer_etiquetas(candidates[0], width, height, len(report["categories"])),
        }
        # La imagen recuperada también debe respetar los grupos de imágenes relacionadas.
        related = {
            r["group_id"]
            for r in report["images"]
            if r["pixel_sha256"] == pixel_hash or (int(r["dhash"], 16) ^ dhash).bit_count() <= 4
        }
        if related:
            group = min(related | {row["group_id"]})
            for previous in report["images"]:
                if previous["group_id"] in related:
                    previous["group_id"] = group
            row["group_id"] = group
        report["images"].append(row)
        recovered.append(
            {
                "image": row["file_name"],
                "original_expected_label": expected.relative_to(root).as_posix(),
                "resolved_label": candidates[0].relative_to(root).as_posix(),
                "reason": "unique_orphan_sidecar_same_stem",
            }
        )
        orphan_paths.remove(str(candidates[0].resolve()))
    if orphan_paths:
        raise ValueError(f"Persisten etiquetas huérfanas no resueltas: {orphan_paths}")
    return recovered




Appending to /content/Tarea2_DL/codigo_parte1/base_parte1.py


In [ ]:
%%writefile -a /content/Tarea2_DL/codigo_parte1/base_parte1.py
def excluir_contradictorios(report):
    """Aparta todas las copias con píxeles idénticos y anotaciones contradictorias."""
    by_pixels = {}
    for row in report["images"]:
        by_pixels.setdefault(row["pixel_sha256"], []).append(row)
    excluded = []
    for same_image in by_pixels.values():
        signatures = {
            tuple(sorted(((a["category_id"], tuple(np.round(a["bbox_xyxy"], 3))) for a in r["annotations"])))
            for r in same_image
        }
        # Se excluyen todas las copias del conflicto; no se decide qué etiqueta es correcta.
        if len(signatures) > 1:
            excluded.extend(
                (
                    {
                        "id": r["id"],
                        "file_name": r["file_name"],
                        "pixel_sha256": r["pixel_sha256"],
                        "annotations": r["annotations"],
                        "reason": "identical_pixels_conflicting_annotations",
                    }
                    for r in same_image
                )
            )
    excluded_ids = {r["id"] for r in excluded}
    report["images"] = [r for r in report["images"] if r["id"] not in excluded_ids]
    return excluded, excluded_ids


def actualizar_resumen(report):
    """Vuelve a contar el dataset después de recuperar etiquetas y excluir conflictos."""
    counts = Counter((a["category_id"] for r in report["images"] for a in r["annotations"]))
    image_counts = Counter(
        (c for r in report["images"] for c in {a["category_id"] for a in r["annotations"]})
    )
    areas = [
        (a["bbox_xyxy"][2] - a["bbox_xyxy"][0])
        * (a["bbox_xyxy"][3] - a["bbox_xyxy"][1])
        / (r["width"] * r["height"])
        for r in report["images"]
        for a in r["annotations"]
    ]
    report["summary"] = {
        "images": len(report["images"]),
        "objects": sum(counts.values()),
        "classes_declared": len(report["categories"]),
        "classes_observed": len(counts),
        "empty_images": sum((not r["annotations"] for r in report["images"])),
        "groups": len({r["group_id"] for r in report["images"]}),
        "exact_duplicate_pairs": len(report["exact_pairs"]),
        "near_duplicate_pairs": len(report["near_pairs"]),
        "box_relative_area_quantiles": np.quantile(areas, [0, 0.25, 0.5, 0.75, 1]).tolist(),
        "class_counts": [
            {**cat, "objects": counts[cat["id"]], "images": image_counts[cat["id"]]}
            for cat in report["categories"]
        ],
    }


def depurar(source, output):
    """Aplica la política de depuración y deja trazabilidad; los originales quedan intactos."""
    report = copy.deepcopy(source)
    root, output = Path(source["root"]), Path(output)
    recovered = recuperar_etiquetas(report, source, root)
    allowed_prefixes = ("Etiqueta ausente:", "Duplicados con etiquetas diferentes:")
    unexplained = [
        e for e in source["errors"] if not e.startswith(allowed_prefixes) and "etiquetas huérfanas" not in e
    ]
    if unexplained:
        raise ValueError(f"Errores que requieren revisión: {unexplained}")
    excluded, excluded_ids = excluir_contradictorios(report)
    report["source_errors"] = source["errors"]
    report["errors"], report["orphan_labels"], report["valid"] = ([], [], True)
    report["curation"] = {
        "policy": "unique-sidecar reassociation; quarantine every conflicting-pixel duplicate",
        "recovered_labels": recovered,
        "excluded_images": excluded,
        "originals_modified": False,
    }
    report["warnings"] += [
        f"Curación explícita: {len(recovered)} sidecars reasociados y {len(excluded)} imágenes en cuarentena; originales intactos"
    ]
    report["exact_pairs"] = [p for p in source["exact_pairs"] if not set(p) & excluded_ids]
    report["near_pairs"] = [p for p in source["near_pairs"] if not set(p) & excluded_ids]
    actualizar_resumen(report)
    output.mkdir(parents=True, exist_ok=True)
    guardar_json(output / "audit.json", report)
    guardar_json(output / "curation.json", report["curation"])
    guardar_json(output / "catalog.json", report["categories"])
    mostrar_muestra(report, output / "annotations.jpg")
    graficar_datos(report, output)
    return report


# La huella reúne clases, contenido, cajas y grupos; no depende de la ruta absoluta del equipo.
def huella_dataset(auditoria):
    content = {
        "categories": auditoria["categories"],
        "images": [
            (r["file_name"], r["sha256"], r["annotations"], r["group_id"]) for r in auditoria["images"]
        ],
    }
    return hashlib.sha256(__import__("json").dumps(content, sort_keys=True).encode()).hexdigest()


# Impide mezclar particiones antiguas con un catálogo o unas anotaciones diferentes.
def revisar_manifiesto(auditoria, manifest):
    if not auditoria["valid"] or huella_dataset(auditoria) != manifest["dataset_fingerprint"]:
        raise ValueError("La auditoría y las particiones no corresponden al mismo dataset/catálogo")
    return revisar_particiones(auditoria, manifest["partitions"])


# COCO espera x, y, ancho y alto; internamente usamos x1, y1, x2, y2.
def formato_coco(auditoria, image_ids):
    chosen = set(image_ids)
    images, annotations = ([], [])
    for row in auditoria["images"]:
        if row["id"] not in chosen:
            continue
        images.append({k: row[k] for k in ("id", "file_name", "width", "height")})
        for ann in row["annotations"]:
            x1, y1, x2, y2 = ann["bbox_xyxy"]
            annotations.append(
                {
                    "id": len(annotations) + 1,
                    "image_id": row["id"],
                    "category_id": ann["category_id"],
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "area": (x2 - x1) * (y2 - y1),
                    "iscrowd": 0,
                }
            )
    return {
        "info": {"description": "FjordFish common evaluation"},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": auditoria["categories"],
    }


def asignar_grupos(auditoria, ratios, seed):
    """Asigna grupos completos buscando equilibrar imágenes y presencia de clases."""
    groups = defaultdict(list)
    for row in auditoria["images"]:
        groups[row["group_id"]].append(row)
    if len(groups) < 3:
        raise ValueError("Hay menos de tres grupos independientes; revisar auditoría")
    c = len(auditoria["categories"])
    vectors = {
        g: np.bincount([a["category_id"] for row in rows for a in row["annotations"]], minlength=c)
        for g, rows in groups.items()
    }
    total = sum(vectors.values(), np.zeros(c))
    support = np.sum([v > 0 for v in vectors.values()], axis=0)
    # La semilla resuelve empates de manera repetible.
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    keys.sort(
        key=lambda g: (
            min((support[i] for i in np.flatnonzero(vectors[g])), default=len(groups) + 1),
            -len(groups[g]),
        )
    )
    target_n = np.asarray(ratios) * len(auditoria["images"])
    target_c = np.asarray(ratios)[:, None] * total
    sizes, class_counts = (np.zeros(3), np.zeros((3, c)))
    assigned = [[], [], []]
    for pos, g in enumerate(keys):
        n, v = (len(groups[g]), vectors[g])
        empty = [i for i in range(3) if not assigned[i]]
        if len(keys) - pos == len(empty):
            candidates = empty
        # Una clase presente en un solo grupo se prioriza para entrenamiento.
        elif any((support[i] == 1 for i in np.flatnonzero(v))):
            candidates = [0]
        else:
            candidates = range(3)
        # Balancear cantidad de imágenes y objetos sin romper los grupos.
        costs = []
        for index in candidates:
            image_delta = (
                (sizes[index] + n - target_n[index]) ** 2 - (sizes[index] - target_n[index]) ** 2
            ) / max(target_n[index], 1)
            class_delta = np.sum(
                (
                    (class_counts[index] + v - target_c[index]) ** 2
                    - (class_counts[index] - target_c[index]) ** 2
                )
                / np.maximum(total, 1)
            )
            costs.append((float(image_delta + 0.3 * class_delta), index))
        _, index = min(costs)
        assigned[index].extend((row["id"] for row in groups[g]))
        sizes[index] += n
        class_counts[index] += v
    names = ["train", "val", "test"]
    partitions = dict(zip(names, [sorted(ids) for ids in assigned]))
    return partitions




Appending to /content/Tarea2_DL/codigo_parte1/base_parte1.py


In [ ]:
%%writefile -a /content/Tarea2_DL/codigo_parte1/base_parte1.py
def exportar_particiones(auditoria, partitions, output):
    """Genera COCO y YOLO con las mismas imágenes y las etiquetas corregidas."""
    c = len(auditoria["categories"])
    names = list(partitions)
    by_id = {r["id"]: r for r in auditoria["images"]}
    stats = {}
    export_counts = {"hardlinked": 0, "copied": 0, "reused": 0}
    for name, ids in partitions.items():
        guardar_json(output / f"{name}.coco.json", formato_coco(auditoria, ids))
        import os
        import shutil

        paths = []
        for identifier in ids:
            row = by_id[identifier]
            source = Path(auditoria["root"]) / row["file_name"]
            image_path = output / "yolo/images" / name / f"{identifier:06d}{source.suffix.lower()}"
            labels_path = output / "yolo/labels" / name / f"{identifier:06d}.txt"
            image_path.parent.mkdir(parents=True, exist_ok=True)
            labels_path.parent.mkdir(parents=True, exist_ok=True)
            if image_path.exists() and (not image_path.samefile(source)):
                if calcular_sha256(image_path) != row["sha256"]:
                    image_path.unlink()
            if not image_path.exists():
                try:
                    # El enlace duro evita duplicar las imágenes; si falla, se copia el archivo.
                    os.link(source, image_path)
                    export_counts["hardlinked"] += 1
                except OSError:
                    shutil.copy2(source, image_path)
                    export_counts["copied"] += 1
            else:
                export_counts["reused"] += 1
            # Exportar las cajas de la auditoría depurada, no las etiquetas originales conflictivas.
            labels_path.write_text(
                "\n".join(
                    (
                        str(a["category_id"])
                        + " "
                        + " ".join(
                            (f"{v:.10f}" for v in cajas_a_yolo(a["bbox_xyxy"], row["width"], row["height"]))
                        )
                        for a in row["annotations"]
                    )
                )
                + "\n",
                encoding="utf-8",
            )
            paths.append(image_path.resolve().as_posix())
        (output / f"{name}.txt").write_text("\n".join(paths) + "\n", encoding="utf-8")
        class_objects = Counter((a["category_id"] for i in ids for a in by_id[i]["annotations"]))
        stats[name] = {
            "images": len(ids),
            "groups": len({by_id[i]["group_id"] for i in ids}),
            "objects_by_class": {str(k): class_objects[k] for k in range(c)},
            "classes_without_gt": [k for k in range(c) if not class_objects[k]],
        }
    data = {
        "path": str((output / "yolo").resolve()),
        **{n: str((output / f"{n}.txt").resolve()) for n in names},
        "names": {cat["id"]: cat["name"] for cat in auditoria["categories"]},
        "nc": c,
    }
    (output / "data.yaml").write_text(
        yaml.safe_dump(data, allow_unicode=True, sort_keys=False), encoding="utf-8"
    )
    return stats, export_counts


def crear_particiones(auditoria, output, ratios=(0.7, 0.15, 0.15), seed=42):
    """Valida la entrada, crea la división común y guarda su huella reproducible."""
    if not auditoria["valid"]:
        raise ValueError("La auditoría tiene errores: no se crearán particiones")
    if len(ratios) != 3 or any((x <= 0 for x in ratios)) or abs(sum(ratios) - 1) > 1e-08:
        raise ValueError("Se requieren tres proporciones positivas que sumen uno")
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    partitions = asignar_grupos(auditoria, ratios, seed)
    revisar_particiones(auditoria, partitions)
    stats, export_counts = exportar_particiones(auditoria, partitions, output)
    fingerprint = huella_dataset(auditoria)
    manifest = {
        "schema_version": 1,
        "seed": seed,
        "ratios_requested": list(ratios),
        "dataset_fingerprint": fingerprint,
        "partitions": partitions,
        "statistics": stats,
        "warnings": auditoria["warnings"],
        "yolo_export": export_counts,
        "method": "greedy grouped image/class balance; singleton classes prefer train",
    }
    guardar_json(output / "manifest.json", manifest)
    return manifest


# Cada imagen y cada grupo deben aparecer en un solo conjunto y cubrir todo el dataset.
def revisar_particiones(auditoria, partitions):
    by_id = {r["id"]: r for r in auditoria["images"]}
    if set(partitions) != {"train", "val", "test"}:
        raise ValueError("Se requieren train, val y test")
    seen_ids, seen_groups = (set(), set())
    for name, ids in partitions.items():
        if not ids or len(ids) != len(set(ids)):
            raise ValueError(f"Partición vacía o IDs repetidos: {name}")
        if set(ids) - set(by_id) or seen_ids & set(ids):
            raise ValueError("IDs desconocidos o compartidos entre particiones")
        current_groups = {by_id[i]["group_id"] for i in ids}
        if seen_groups & current_groups:
            raise ValueError("Filtración de grupos entre particiones")
        seen_ids.update(ids)
        seen_groups.update(current_groups)
    if seen_ids != set(by_id):
        raise ValueError("Hay imágenes no asignadas")
    return True


# Se calcula exclusivamente con train para no incorporar información de validación o prueba.
def calcular_normalizacion(auditoria, manifest, size=512):
    """Estadísticas por píxel de las entradas letterbox, SOLO de train."""
    revisar_manifiesto(auditoria, manifest)
    ids = set(manifest["partitions"]["train"])
    count, sums, squares = (0, np.zeros(3, dtype=np.float64), np.zeros(3, dtype=np.float64))
    for row in auditoria["images"]:
        if row["id"] not in ids:
            continue
        with Image.open(Path(auditoria["root"]) / row["file_name"]) as raw:
            image, _ = ajustar_imagen(raw, size)
        array = np.asarray(image, dtype=np.float64).reshape(-1, 3) / 255
        sums += array.sum(axis=0)
        squares += (array * array).sum(axis=0)
        count += len(array)
    mean = sums / count
    std = np.sqrt(np.maximum(squares / count - mean**2, 1e-12))
    return {
        "mean": mean.tolist(),
        "std": std.tolist(),
        "split": "train",
        "size": size,
        "images": len(ids),
        "pixels": count,
        "dataset_fingerprint": manifest["dataset_fingerprint"],
    }


# Rechaza IDs desconocidos, cajas inválidas y puntajes no finitos antes de medir AP.


Appending to /content/Tarea2_DL/codigo_parte1/base_parte1.py


In [ ]:
%%writefile -a /content/Tarea2_DL/codigo_parte1/base_parte1.py
def validar_predicciones(ground_truth, predictions):
    images = {i["id"]: i for i in ground_truth["images"]}
    classes = {c["id"] for c in ground_truth["categories"]}
    for p in predictions:
        if p["image_id"] not in images or p["category_id"] not in classes:
            raise ValueError("Predicción con imagen o clase desconocida")
        values = np.asarray([*p["bbox"], p["score"]], dtype=float)
        if len(p["bbox"]) != 4 or not np.isfinite(values).all():
            raise ValueError("Predicción no finita o caja mal formada")
        x, y, w, h = p["bbox"]
        image = images[p["image_id"]]
        if (
            w <= 0
            or h <= 0
            or min(x, y) < -0.001
            or (x + w > image["width"] + 0.001)
            or (y + h > image["height"] + 0.001)
        ):
            raise ValueError("Caja predicha fuera de límites o degenerada")
        if not 0 <= p["score"] <= 1:
            raise ValueError("El score debe estar entre cero y uno")


# El emparejamiento es uno a uno: una segunda detección del mismo objeto cuenta como falso positivo.
def calcular_metricas(ground_truth, predictions, threshold=0.5, iou_threshold=0.5):
    validar_predicciones(ground_truth, predictions)
    gt = defaultdict(list)
    for ann in ground_truth["annotations"]:
        if not ann.get("iscrowd", 0):
            x, y, w, h = ann["bbox"]
            gt[ann["image_id"], ann["category_id"]].append([x, y, x + w, y + h])
    matched = defaultdict(set)
    tp = fp = 0
    for p in sorted((p for p in predictions if p["score"] >= threshold), key=lambda p: -p["score"]):
        key = (p["image_id"], p["category_id"])
        x, y, w, h = p["bbox"]
        overlaps = iou_cajas([[x, y, x + w, y + h]], gt[key])[0]
        eligible = [i for i in range(len(overlaps)) if i not in matched[key] and overlaps[i] >= iou_threshold]
        if eligible:
            matched[key].add(max(eligible, key=lambda i: overlaps[i]))
            tp += 1
        else:
            fp += 1
    fn = sum(map(len, gt.values())) - tp
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {
        "threshold": float(threshold),
        "iou": iou_threshold,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
    }


# Elegir el umbral con test contaminaría la evaluación; aquí solo se permite val.
def elegir_umbral(ground_truth, predictions, split):
    if split != "val":
        raise ValueError("La calibración de umbral solo se permite en validación")
    candidates = [calcular_metricas(ground_truth, predictions, t / 100) for t in range(5, 100, 5)]
    best = max(candidates, key=lambda r: (r["f1"], r["threshold"]))
    return {"selected": best, "candidates": candidates, "split": "val"}


# COCOeval calcula AP en varios IoU; las clases sin anotaciones se informan como no definidas.
def evaluar(ground_truth, predictions, threshold=0.5):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    validar_predicciones(ground_truth, predictions)
    if not ground_truth["images"]:
        raise ValueError("El conjunto de evaluación está vacío")
    capture = io.StringIO()
    with redirect_stdout(capture):
        gt = COCO()
        gt.dataset = copy.deepcopy(ground_truth)
        gt.createIndex()
        if predictions:
            dt = gt.loadRes(copy.deepcopy(predictions))
        else:
            dt = COCO()
            dt.dataset = {**copy.deepcopy(ground_truth), "annotations": []}
            dt.createIndex()
        evaluator = COCOeval(gt, dt, "bbox")
        evaluator.params.imgIds = sorted((i["id"] for i in ground_truth["images"]))
        evaluator.params.catIds = sorted((c["id"] for c in ground_truth["categories"]))
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    precision = evaluator.eval["precision"]
    per_class = []
    for k, cat in enumerate(sorted(ground_truth["categories"], key=lambda c: c["id"])):
        values = precision[:, :, k, 0, -1]
        valid = values[values >= 0]
        per_class.append(
            {
                **cat,
                "ap": float(valid.mean()) if valid.size else None,
                "objects": sum((a["category_id"] == cat["id"] for a in ground_truth["annotations"])),
            }
        )
    return {
        "map_50_95": float(evaluator.stats[0]) if evaluator.stats[0] >= 0 else None,
        "ap_50": float(evaluator.stats[1]) if evaluator.stats[1] >= 0 else None,
        "per_class": per_class,
        "operating_point": calcular_metricas(ground_truth, predictions, threshold),
        "max_detections": 100,
        "coco_summary": capture.getvalue(),
        "images": len(ground_truth["images"]),
        "predictions": len(predictions),
    }


# Controla los generadores aleatorios para repetir los pilotos y verificar los checkpoints.
def fijar_semilla(seed=42):
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


# La disponibilidad de CUDA se comprueba con operaciones reales, no solo consultando un nombre de GPU.
def revisar_entorno(output=None, require_gpu=False):
    packages = {}
    for name in ("torch", "torchvision", "ultralytics", "numpy", "Pillow", "PyYAML", "pycocotools", "pytest"):
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = None
    report = {
        "python": sys.version,
        "platform": platform.platform(),
        "packages": packages,
        "cuda_available": False,
        "gpu_test_passed": False,
        "status": "cpu_only",
        "devices": [],
    }
    import torch
    from torchvision.ops import nms

    try:
        nms(torch.tensor([[0.0, 0.0, 1.0, 1.0]]), torch.tensor([0.9]), 0.5)
        report["torchvision_nms_cpu"] = True
        report["cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            for index in range(torch.cuda.device_count()):
                p = torch.cuda.get_device_properties(index)
                report["devices"].append({"name": p.name, "memory_bytes": p.total_memory})
            x = torch.randn(128, 128, device="cuda", requires_grad=True)
            x.square().mean().backward()
            nms(torch.tensor([[0.0, 0.0, 1.0, 1.0]], device="cuda"), torch.tensor([0.9], device="cuda"), 0.5)
            torch.cuda.synchronize()
            report["gpu_test_passed"] = bool(torch.isfinite(x.grad).all())
            report["status"] = "gpu_verified" if report["gpu_test_passed"] else "gpu_failed"
            report["cuda_runtime"] = torch.version.cuda
    except Exception as exc:
        report["status"], report["error"] = ("failed", repr(exc))
    report["timestamp_unix"] = time.time()
    if output:
        guardar_json(output, report)
    if report["status"] == "failed" or (require_gpu and (not report["gpu_test_passed"])):
        raise RuntimeError(f"Entorno no cumple el diagnóstico solicitado: {report}")
    return report


# Los mapas a distintas escalas permiten al cabezal SSD trabajar con objetos de varios tamaños.


Appending to /content/Tarea2_DL/codigo_parte1/base_parte1.py


In [ ]:
%%writefile -a /content/Tarea2_DL/codigo_parte1/base_parte1.py
class ExtractorPropio(nn.Module):
    out_channels = [128, 192, 256, 256]

    def __init__(self):
        super().__init__()
        blocks = []
        previous = 3
        for channels in (32, 64, 128, 192, 256, 256):
            blocks.append(
                nn.Sequential(
                    nn.Conv2d(previous, channels, 3, stride=2, padding=1, bias=False),
                    nn.BatchNorm2d(channels),
                    nn.ReLU(inplace=True),
                    nn.Conv2d(channels, channels, 3, padding=1, bias=False),
                    nn.BatchNorm2d(channels),
                    nn.ReLU(inplace=True),
                )
            )
            previous = channels
        self.blocks = nn.ModuleList(blocks)
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")

    def forward(self, x):
        result = OrderedDict()
        for index, block in enumerate(self.blocks):
            x = block(x)
            if index >= 2:
                result[str(index - 2)] = x
        return result


class ControlDetector(nn.Module):
    """Controla las fases y qué parámetros pueden aprender."""
    def __init__(self, network, kind, num_classes, size, provenance):
        super().__init__()
        self.network = network
        self.kind, self.num_classes, self.size = (kind, num_classes, size)
        self.provenance = provenance
        self.phase = "full"

    def set_phase(self, phase):
        if phase not in {"full", "frozen", "finetune"}:
            raise ValueError("Fase desconocida")
        self.phase = phase
        for parameter in self.network.parameters():
            parameter.requires_grad_(True)
        if self.kind.startswith("yolo"):
            for module in self.network.model:
                frozen = phase == "frozen" and module.i <= 10
                for parameter in module.parameters():
                    parameter.requires_grad_(not frozen)
            for name, parameter in self.network.named_parameters():
                if ".dfl." in name:
                    parameter.requires_grad_(False)
        elif self.kind == "fasterrcnn_coco":
            for name, parameter in self.network.backbone.body.named_parameters():
                parameter.requires_grad_(
                    phase == "full" or (phase == "finetune" and name.startswith("layer4."))
                )
        self.train(self.training)
        return self.parameter_report()

    def train(self, mode=True):
        super().train(mode)
        # requires_grad=False no basta: BatchNorm también actualiza estadísticas en modo train.
        if mode and self.phase == "frozen" and self.kind.startswith("yolo"):
            for module in self.network.model:
                if module.i <= 10:
                    module.eval()
        return self

    def parameter_report(self):
        return {
            "total": sum((p.numel() for p in self.parameters())),
            "trainable": sum((p.numel() for p in self.parameters() if p.requires_grad)),
            "trainable_names": [n for n, p in self.named_parameters() if p.requires_grad],
            "frozen_names": [n for n, p in self.named_parameters() if not p.requires_grad],
            "phase": self.phase,
            "provenance": self.provenance,
        }


class Detector(ControlDetector):
    """Adapta pérdidas y predicciones a una interfaz común para los cuatro modelos."""
    def loss(self, images, targets):
        self.train()
        # Torchvision reserva la clase 0 para fondo; nuestro catálogo comienza en 0.
        if not self.kind.startswith("yolo"):
            native = [{"boxes": t["boxes"], "labels": t["labels"] + 1} for t in targets]
            losses = self.network(images, native)
            return (sum(losses.values()), losses)
        image_batch = torch.stack(images)
        indices, labels, boxes = ([], [], [])
        for index, target in enumerate(targets):
            b = target["boxes"]
            xywh = torch.cat(((b[:, :2] + b[:, 2:]) / 2, b[:, 2:] - b[:, :2]), dim=1) / self.size
            boxes.append(xywh)
            labels.append(target["labels"].reshape(-1, 1).float())
            indices.append(torch.full((len(b),), index, device=b.device))
        batch = {
            "img": image_batch,
            "batch_idx": torch.cat(indices),
            "cls": torch.cat(labels),
            "bboxes": torch.cat(boxes),
        }
        loss, components = self.network(batch)
        return (loss.sum(), components)

    @torch.no_grad()
    def predict(self, images):
        self.eval()
        device = next(self.parameters()).device
        prepared, mappings = ([], [])
        for image in images:
            processed, mapping = ajustar_imagen(image, self.size)
            prepared.append(
                torch.from_numpy(np.asarray(processed).copy()).permute(2, 0, 1).float().div(255).to(device)
            )
            mappings.append(mapping)
        if not prepared:
            return []
        # YOLO y Torchvision decodifican distinto; ambas salidas se llevan al mismo formato.
        if self.kind.startswith("yolo"):
            predictions = self.network(torch.stack(prepared))
            decoded = predictions[0] if isinstance(predictions, tuple) else predictions
            candidates = []
            for raw in decoded:
                raw = raw.detach().float().cpu().numpy()
                location, label = np.where(raw[4:].T >= 0.001)
                xywh = raw[:4, location].T
                boxes = np.concatenate((xywh[:, :2] - xywh[:, 2:] / 2, xywh[:, :2] + xywh[:, 2:] / 2), axis=1)
                candidates.append((boxes, label, raw[4 + label, location]))
        else:
            candidates = [
                (p["boxes"].cpu().numpy(), p["labels"].cpu().numpy() - 1, p["scores"].cpu().numpy())
                for p in self.network(prepared)
            ]
        result = []
        for (boxes, labels, scores), mapping in zip(candidates, mappings):
            # Evaluar en coordenadas originales y aplicar el mismo filtro por clase.
            boxes = mapping.inverse_boxes(boxes)
            boxes, labels, scores = filtrar_detecciones(boxes, labels, scores)
            result.append({"boxes": boxes.tolist(), "labels": labels.tolist(), "scores": scores.tolist()})
        return result




Appending to /content/Tarea2_DL/codigo_parte1/base_parte1.py


In [ ]:
%%writefile -a /content/Tarea2_DL/codigo_parte1/base_parte1.py
def crear_cnn_ssd(num_classes, size, statistics):
    """Combina el extractor propio con los cabezales y las pérdidas SSD de Torchvision."""
    provenance = {"initialization": "random", "pretrained": False}
    mean = statistics["mean"] if statistics else [0.5] * 3
    std = statistics["std"] if statistics else [0.25] * 3
    network = SSD(
        ExtractorPropio(),
        DefaultBoxGenerator([[2, 3]] * 4, scales=[0.04, 0.1, 0.25, 0.55, 0.9]),
        (size, size),
        num_classes + 1,
        image_mean=mean,
        image_std=std,
        score_thresh=0.001,
        nms_thresh=0.5,
        detections_per_img=100,
        topk_candidates=400,
    )
    provenance["normalization"] = "train" if statistics else "synthetic_smoke_defaults"
    return network, provenance


def crear_yolo(kind, num_classes, weights_dir):
    """Usa la misma arquitectura YOLO11n para estudiar el efecto del preentrenamiento."""
    provenance = {"initialization": "random", "pretrained": False}
    config_root = Path(os.environ.get("YOLO_CONFIG_DIR", "resultados_parte_1/ultralytics")).resolve()
    (config_root / "Ultralytics").mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("YOLO_CONFIG_DIR", str(config_root))
    os.environ.setdefault("YOLO_AUTOINSTALL", "false")
    import ultralytics
    import yaml
    from ultralytics.nn.tasks import DetectionModel
    from ultralytics.cfg import get_cfg

    if ultralytics.__version__ != "8.4.174":
        raise RuntimeError("Esta implementación requiere ultralytics==8.4.174; instala esa versión y reinicia el kernel")
    cfg = yaml.safe_load((Path(ultralytics.__file__).parent / "cfg/models/11/yolo11.yaml").read_text())
    cfg["scale"], cfg["nc"] = ("n", num_classes)
    network = DetectionModel(cfg, nc=num_classes, verbose=False)
    network.args = get_cfg()
    if kind != "yolo_scratch":
        raise ValueError("Parte 1: solo YOLO desde cero, sin checkpoints COCO")
    return network, provenance


def crear_detector(kind, num_classes, size=512, weights_dir="resultados_etapas_1_2/pesos", statistics=None):
    """Construye el modelo pedido y configura su fase inicial de entrenamiento."""
    if kind not in KINDS or num_classes < 1 or size < 64 or size % 64:
        raise ValueError("Modelo/clases inválidos o tamaño no múltiplo de 64 (mínimo 64)")
    if kind == "cnn_ssd":
        network, provenance = crear_cnn_ssd(num_classes, size, statistics)
    elif kind.startswith("yolo"):
        network, provenance = crear_yolo(kind, num_classes, weights_dir)
    else:
        raise ValueError("Esta copia solo admite modelos desde cero")
    detector = Detector(network, kind, num_classes, size, provenance)
    detector.set_phase("full" if kind in {"cnn_ssd", "yolo_scratch"} else "frozen")
    return detector


# La misma transformación se aplica a la imagen y a las cajas; el piloto desactiva aumentos.
class DatosDeteccion(Dataset):

    def __init__(self, auditoria, image_ids, size=512, augment=False):
        ids = set(image_ids)
        self.rows = [r for r in auditoria["images"] if r["id"] in ids]
        if len(self.rows) != len(ids):
            raise ValueError("IDs desconocidos en el dataset")
        self.root, self.size, self.augment = (Path(auditoria["root"]), size, augment)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(self.root / row["file_name"]) as raw:
            if raw.size != (row["width"], row["height"]):
                raise ValueError("La imagen cambió de dimensiones después de la auditoría")
            image = raw.convert("RGB")
        boxes = np.asarray([a["bbox_xyxy"] for a in row["annotations"]], dtype=np.float32).reshape(-1, 4)
        labels = torch.tensor([a["category_id"] for a in row["annotations"]], dtype=torch.int64)
        if self.augment and random.random() < 0.5:
            image, boxes = reflejar_imagen(image, boxes)
        image, mapping = ajustar_imagen(image, self.size)
        boxes = torch.tensor(mapping.forward_boxes(boxes), dtype=torch.float32)
        tensor = torch.from_numpy(np.asarray(image).copy()).permute(2, 0, 1).float() / 255
        return (tensor, {"boxes": boxes, "labels": labels, "image_id": torch.tensor(row["id"])})


# Guarda pesos, optimizador y estado aleatorio para poder retomar el siguiente paso.
def guardar_checkpoint(path, detector, optimizer, step, huella_dataset, training=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    numpy_state = np.random.get_state()
    # Incluir los generadores evita cambiar el siguiente lote aleatorio al reanudar.
    state = {
        "schema_version": 1,
        "model": detector.state_dict(),
        "optimizer": optimizer.state_dict(),
        "kind": detector.kind,
        "num_classes": detector.num_classes,
        "size": detector.size,
        "phase": detector.phase,
        "step": step,
        "dataset_fingerprint": huella_dataset,
        "rng": {
            "python": random.getstate(),
            "torch": torch.get_rng_state(),
            "numpy": [numpy_state[0], numpy_state[1].tolist(), *numpy_state[2:]],
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
        },
    }
    if training is not None:
        state["training"] = training
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(state, temporary)
    temporary.replace(path)


# Comprueba la identidad del experimento antes de restaurar el entrenamiento.
def recuperar_checkpoint(path, detector, optimizer, huella_dataset, state=None):
    device = next(detector.parameters()).device
    if state is None:
        state = torch.load(path, map_location=device, weights_only=True)
    for key, expected in {
        "kind": detector.kind,
        "num_classes": detector.num_classes,
        "size": detector.size,
        "phase": detector.phase,
        "dataset_fingerprint": huella_dataset,
    }.items():
        if state[key] != expected:
            raise ValueError(f"Checkpoint incompatible: {key}")
    detector.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    rng = state["rng"]
    random.setstate(rng["python"])
    npstate = rng["numpy"]
    np.random.set_state((npstate[0], np.asarray(npstate[1], dtype=np.uint32), *npstate[2:]))
    torch.set_rng_state(rng["torch"].cpu())
    if rng["cuda"]:
        if not torch.cuda.is_available() or len(rng["cuda"]) != torch.cuda.device_count():
            raise ValueError("Para restaurar RNG CUDA se requiere la misma cantidad de GPUs")
        torch.cuda.set_rng_state_all([x.cpu() for x in rng["cuda"]])
    return {
        key: state[key] for key in ("kind", "num_classes", "size", "phase", "step", "dataset_fingerprint")
    }


# El decaimiento se aplica a matrices de pesos; se excluyen sesgos y parámetros unidimensionales.


Appending to /content/Tarea2_DL/codigo_parte1/base_parte1.py


In [ ]:
%%writefile -a /content/Tarea2_DL/codigo_parte1/base_parte1.py
def adamw(detector, learning_rate=0.001, weight_decay=0.0001):
    groups = {True: [], False: []}
    for name, parameter in detector.named_parameters():
        if parameter.requires_grad:
            groups[parameter.ndim > 1 and (not name.endswith("bias"))].append(parameter)
    return torch.optim.AdamW(
        [{"params": groups[True], "weight_decay": weight_decay}, {"params": groups[False], "weight_decay": 0.0}],
        lr=learning_rate,
    )


# Pruebas y pilotos comparten este paso; una pérdida o gradiente no finito detiene el flujo.
def paso_entrenamiento(detector, optimizer, images, targets):
    """Un solo paso compartido por pruebas, pilotos y fine-tuning."""
    optimizer.zero_grad(set_to_none=True)
    loss, _ = detector.loss(images, targets)
    if not torch.isfinite(loss):
        raise RuntimeError("La pérdida no es finita")
    loss.backward()
    gradients = [p.grad for p in detector.parameters() if p.requires_grad and p.grad is not None]
    if not gradients or not all((bool(torch.isfinite(g).all()) for g in gradients)):
        raise RuntimeError("Faltan gradientes o hay valores no finitos")
    optimizer.step()
    return float(loss.detach())


# Congelar implica mantener los pesos y las estadísticas que deberían permanecer fijos.
def revisar_congelados(detector, saved):
    current = {**dict(detector.named_parameters()), **dict(detector.named_buffers())}
    if any((not torch.equal(value, current[name]) for name, value in saved.items())):
        raise RuntimeError("Cambió un peso o una estadística que debía estar congelada")


# Las predicciones ya están en la resolución original cuando se convierten a COCO.
def predicciones_coco(image_ids, outputs):
    result = []
    for image_id, output in zip(image_ids, outputs):
        for box, label, score in zip(output["boxes"], output["labels"], output["scores"]):
            x1, y1, x2, y2 = box
            result.append(
                {
                    "image_id": image_id,
                    "category_id": label,
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": score,
                }
            )
    return result


def comprobar_un_modelo(
    kind, num_classes, size, weights_dir, statistics, device, steps
):
    """Revisa actualización de pesos, inferencia, congelación y recuperación exacta."""
    detector = crear_detector(kind, num_classes, size, weights_dir, statistics).to(
        device
    )
    optimizer = adamw(detector)
    images = [torch.rand(3, size, size, device=device) for _ in range(2)]
    targets = [
        {
            "boxes": torch.tensor([[0.2, 0.2, 0.7, 0.7]], device=device) * size,
            "labels": torch.tensor([0], device=device),
        }
        for _ in images
    ]
    # Conservar una copia permite demostrar que los parámetros congelados no cambiaron.
    frozen = {
        n: p.detach().clone()
        for n, p in detector.named_parameters()
        if not p.requires_grad
    }
    if kind == "yolo_coco":
        frozen.update(
            {
                n: b.clone()
                for n, b in detector.named_buffers()
                if "running_" in n and int(n.split(".")[2]) <= 10
            }
        )
    name, parameter = next(
        ((n, p) for n, p in detector.named_parameters() if p.requires_grad)
    )
    before = parameter.detach().clone()
    losses = [
        paso_entrenamiento(detector, optimizer, images, targets) for _ in range(steps)
    ]
    revisar_congelados(detector, frozen)
    if torch.equal(before, parameter):
        raise RuntimeError("Los pesos entrenables no se actualizaron")
    original = Image.new("RGB", (143, 87), (90, 90, 90))
    predictions = detector.predict([original])
    validar_predicciones(
        {
            "images": [{"id": 1, "width": 143, "height": 87}],
            "categories": [{"id": i} for i in range(num_classes)],
        },
        predicciones_coco([1], predictions),
    )
    # Alterar un peso y restaurar el checkpoint comprueba una recuperación real.
    with tempfile.TemporaryDirectory() as temporary:
        path = Path(temporary) / "checkpoint.pt"
        guardar_checkpoint(path, detector, optimizer, steps, "prueba-sintetica")
        with torch.no_grad():
            parameter.add_(1)
        recuperar_checkpoint(path, detector, optimizer, "prueba-sintetica")
        if detector.predict([original]) != predictions:
            raise RuntimeError("La recuperación cambió las predicciones")
    # Una imagen sin objetos no debería provocar errores ni valores no finitos.
    empty = [
        {
            "boxes": torch.empty(0, 4, device=device),
            "labels": torch.empty(0, dtype=torch.long, device=device),
        }
        for _ in images
    ]
    empty_loss = paso_entrenamiento(detector, optimizer, images, empty)
    details = {
        "passed": True,
        "losses": losses,
        "empty_loss": empty_loss,
        "checkpoint_restored": True,
        "frozen_unchanged": True,
        "trainable_updated": True,
        "parameters": detector.parameter_report(),
    }
    # Al habilitar fine-tuning, verificar que el backbone vuelva a actualizarse.
    if kind.endswith("coco"):
        detector.set_phase("finetune")
        name, parameter = next(
            (
                (n, p)
                for n, p in detector.named_parameters()
                if p.requires_grad and n in frozen
            )
        )
        before = parameter.detach().clone()
        details["finetune_loss"] = paso_entrenamiento(
            detector, adamw(detector, 0.0001), images, targets
        )
        if torch.equal(before, parameter):
            raise RuntimeError("Fine-tuning no actualizó el backbone")
        details["finetune_updated"] = name
    return details



Appending to /content/Tarea2_DL/codigo_parte1/base_parte1.py


In [ ]:
%%writefile -a /content/Tarea2_DL/codigo_parte1/base_parte1.py
def probar_modelos(
    output,
    kinds=KINDS,
    weights_dir="resultados_etapas_1_2/pesos",
    device="cpu",
    size=128,
    steps=2,
    num_classes=2,
    statistics=None,
):
    """Comprueba entrenamiento, congelación, predicción y recuperación."""
    if steps < 1 or (device.startswith("cuda") and (not torch.cuda.is_available())):
        raise ValueError("Revisar los pasos o la disponibilidad de CUDA")
    report = {
        "scope": "synthetic_integration_not_fjordfish_training",
        "device": device,
        "size": size,
        "steps": steps,
        "models": {},
        "passed": True,
    }
    for kind in kinds:
        fijar_semilla()
        started = time.perf_counter()
        print("Prueba breve:", kind, flush=True)
        try:
            details = comprobar_un_modelo(kind, num_classes, size, weights_dir, statistics, device, steps)
        except Exception as error:
            details = {"passed": False, "error": str(error)}
            report["passed"] = False
        report["models"][kind] = {**details, "seconds": time.perf_counter() - started}
        guardar_json(output, report)
    return report


# Los datos sintéticos tienen respuestas conocidas para comprobar geometría y métricas.
def crear_datos_prueba(root, count=40, seed=123):
    root = Path(root)
    (root / "images").mkdir(parents=True, exist_ok=True)
    (root / "labels").mkdir(exist_ok=True)
    (root / "data.yaml").write_text("names: [rectangle, ellipse]\nnc: 2\n", encoding="utf-8")
    rng = np.random.default_rng(seed)
    for index in range(count):
        width, height = (96 + index % 3 * 16, 64 + index % 4 * 8)
        pixels = rng.integers(0, 70, (height, width, 3), dtype=np.uint8)
        image = Image.fromarray(pixels)
        draw = ImageDraw.Draw(image)
        labels = []
        for obj in range(0 if index % 11 == 0 else 1 + index % 2):
            x1, y1 = (8 + obj * width // 2, 8 + index % 5)
            x2, y2 = (min(width - 2, x1 + width // 3), min(height - 2, y1 + height // 2))
            category = (index + obj) % 2
            method = draw.rectangle if category == 0 else draw.ellipse
            method((x1, y1, x2, y2), fill=(220, 50, 40) if category == 0 else (40, 220, 50))
            labels.append(
                f"{category} {(x1 + x2) / (2 * width):.8f} {(y1 + y2) / (2 * height):.8f} {(x2 - x1) / width:.8f} {(y2 - y1) / height:.8f}"
            )
        image.save(root / "images" / f"{index:04d}.png")
        (root / "labels" / f"{index:04d}.txt").write_text("\n".join(labels), encoding="utf-8")
    return root


# Mide cobertura geométrica antes de aprender regresión; esta cifra no es el recall del detector.
def revisar_anclas(auditoria, manifest, size=512):
    revisar_manifiesto(auditoria, manifest)
    if size % 64:
        raise ValueError("El tamaño debe ser múltiplo de 64")
    generator = DefaultBoxGenerator([[2, 3]] * 4, scales=[0.04, 0.1, 0.25, 0.55, 0.9])
    features = [torch.zeros(1, 1, size // stride, size // stride) for stride in (8, 16, 32, 64)]
    anchors = generator(ImageList(torch.zeros(1, 3, size, size), [(size, size)]), features)[0].numpy()
    ids = set(manifest["partitions"]["train"])
    values, categories = ([], [])
    for row in auditoria["images"]:
        if row["id"] not in ids or not row["annotations"]:
            continue
        _, mapping = ajustar_imagen(Image.new("RGB", (row["width"], row["height"])), size)
        boxes = mapping.forward_boxes([a["bbox_xyxy"] for a in row["annotations"]])
        values.extend(iou_cajas(boxes, anchors).max(axis=1).tolist())
        categories.extend((a["category_id"] for a in row["annotations"]))
    values, categories = (np.asarray(values), np.asarray(categories))
    if not len(values):
        raise ValueError("No hay cajas de entrenamiento")
    return {
        "split": "train",
        "size": size,
        "anchors": len(anchors),
        "objects": len(values),
        "dataset_fingerprint": manifest["dataset_fingerprint"],
        "mean_best_iou": float(values.mean()),
        "recall_iou_50": float((values >= 0.5).mean()),
        "recall_iou_75": float((values >= 0.75).mean()),
        "best_iou_quantiles": np.quantile(values, [0, 0.25, 0.5, 0.75, 1]).tolist(),
        "per_class": [
            {
                **cat,
                "objects": int((categories == cat["id"]).sum()),
                "recall_iou_50": (
                    float((values[categories == cat["id"]] >= 0.5).mean())
                    if (categories == cat["id"]).any()
                    else None
                ),
            }
            for cat in auditoria["categories"]
        ],
    }


Appending to /content/Tarea2_DL/codigo_parte1/base_parte1.py


### 4.2 Gestión y extracción del dataset — `descarga_fjordfish.py` (2 bloques)


In [ ]:
%%writefile /content/Tarea2_DL/codigo_parte1/descarga_fjordfish.py
"""Descarga reanudable, checksum y extracción del ZIP de FjordFish (sin PyTorch)."""
from pathlib import Path
import hashlib
import json
import re
import shutil
import stat
import time
import urllib.error
import urllib.request
import zipfile

RECORD = "17950781"
ARCHIVE_NAME = "FjordFish.zip"
PUBLISHED_MD5 = "951bf139c3e412fa7924fa7ffc34a307"
PUBLISHED_NAMES = [
    "ballan", "cuckoo_female", "cuckoo_male", "corkwing_female", "corkwing_male",
    "goldsinny", "rock_cook", "labrids_unknown", "cod", "cod_yoy", "haddock", "pollack",
    "poor_cod", "saithe", "whiting", "gadids_unknown", "spiny_dogfish", "sharks_unknown",
    "brown_crab", "green_crab", "hermit", "lobster", "decapods_unknown", "ling",
    "two_spot_goby", "black_goby", "sand_goby",
]

def json_atomico(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)

def hash_archivo(path, algorithm="sha256"):
    digest = hashlib.new(algorithm)
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def solicitar(url, headers=None):
    request = urllib.request.Request(url, headers={"User-Agent": "FjordFish-course/parte1", **(headers or {})})
    return urllib.request.urlopen(request, timeout=120)

def entrada_oficial():
    """La alternativa directa también exige el MD5 publicado por el usuario/Zenodo."""
    try:
        with solicitar(f"https://zenodo.org/api/records/{RECORD}") as response:
            metadata = json.load(response)
        entry = next(e for e in metadata["files"] if e["key"] == ARCHIVE_NAME)
    except (OSError, ValueError, KeyError, StopIteration) as error:
        print(f"API de Zenodo no disponible ({error}). Se usará la URL directa y el MD5 publicado.")
        return {"url": f"https://zenodo.org/records/{RECORD}/files/{ARCHIVE_NAME}?download=1", "size": None,
                "checksum": "md5:" + PUBLISHED_MD5, "metadata_source": "published_record_fallback"}
    if entry["checksum"] != "md5:" + PUBLISHED_MD5:
        raise ValueError("El checksum del archivo oficial cambió. Revisar la versión antes de descargar.")
    return {"url": entry["links"]["self"], "size": int(entry["size"]),
            "checksum": entry["checksum"], "metadata_source": "zenodo_api"}

def verificar_zip(path, entry):
    path = Path(path)
    if entry.get("size") is not None and path.stat().st_size != entry["size"]:
        raise ValueError(f"ZIP incompleto: {path.stat().st_size} bytes; esperados {entry['size']}")
    algorithm, expected = entry["checksum"].split(":", 1)
    if hash_archivo(path, algorithm) != expected:
        raise ValueError(f"Checksum incorrecto: {path}. No se extraerá. Conserva el archivo para revisarlo.")

def descargar_reanudable(path, entry, retries=4, opener=solicitar, pause=time.sleep):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        verificar_zip(path, entry)
        return path
    partial = path.with_name(path.name + ".part")
    expected_size = entry.get("size")
    for attempt in range(retries):
        offset = partial.stat().st_size if partial.exists() else 0
        if expected_size is not None and offset > expected_size:
            raise ValueError("El parcial es mayor que el archivo oficial; revisa ese archivo antes de continuar.")
        if offset and expected_size == offset:
            verificar_zip(partial, entry)
            partial.replace(path)
            return path
        if expected_size and shutil.disk_usage(path.parent).free < expected_size - offset:
            raise OSError("No hay espacio para completar el ZIP en la carpeta indicada")
        try:
            headers = {"Range": f"bytes={offset}-"} if offset else {}
            with opener(entry["url"], headers) as response:
                status = response.status
                if status == 206:
                    match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", response.headers.get("Content-Range", ""))
                    if not match or int(match[1]) != offset:
                        raise ValueError("Zenodo devolvió un rango distinto al solicitado")
                    total = int(match[3])
                    if int(match[2]) < offset or int(match[2]) >= total:
                        raise ValueError("Rango HTTP incoherente")
                    if expected_size and total != expected_size:
                        raise ValueError("El tamaño del servidor no coincide con el archivo oficial")
                elif status == 200:
                    # Si el servidor ignora Range, recomenzar evita concatenar dos ZIP.
                    offset = 0
                    length = response.headers.get("Content-Length")
                    total = int(length) if length else expected_size
                else:
                    raise OSError(f"Respuesta inesperada: HTTP {status}")
                if expected_size and total and total != expected_size:
                    raise ValueError("Content-Length no coincide con el tamaño oficial")
                if total and shutil.disk_usage(path.parent).free + (partial.stat().st_size if partial.exists() and offset == 0 else 0) < total - offset:
                    raise OSError("No hay espacio para completar el ZIP")
                completed, next_log = offset, offset + 128 * 1024 * 1024
                with partial.open("ab" if offset else "wb") as stream:
                    while True:
                        chunk = response.read(4 * 1024 * 1024)
                        if not chunk:
                            break
                        stream.write(chunk)
                        completed += len(chunk)
                        if completed >= next_log:
                            print(f"Descarga: {completed / total:.1%}" if total else f"Descargados {completed / 1024**2:.0f} MiB", flush=True)
                            next_log = completed + 128 * 1024 * 1024
                if total and completed != total:
                    raise OSError("Descarga interrumpida; se conserva el parcial para reanudar")
            verificar_zip(partial, entry)
            partial.replace(path)
            return path
        except (urllib.error.URLError, TimeoutError, ConnectionError, OSError) as error:
            if attempt + 1 == retries:
                raise RuntimeError(f"Descarga incompleta tras {retries} intentos. Vuelve a ejecutar para reanudar: {partial}") from error
            print(f"Intento {attempt + 1}: {error}. Se reintentará conservando el parcial.", flush=True)
            pause(min(2 ** attempt, 15))
    raise RuntimeError("No se completó la descarga")

def extraer_seguro(archive, destination, checksum):
    archive, destination = Path(archive), Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    identity = {"record": RECORD, "checksum": checksum, "archive_size": archive.stat().st_size}
    marker, pending = destination / ".extraction.json", destination / ".extraction_pending.json"
    if marker.exists():
        if json.loads(marker.read_text()) != identity:
            raise ValueError("La extracción existente pertenece a otro ZIP")
        return destination
    if pending.exists():
        if json.loads(pending.read_text()) != identity:
            raise ValueError("La extracción interrumpida pertenece a otro ZIP")
    elif any(destination.iterdir()):
        raise ValueError("La carpeta de extracción ya contiene archivos sin marcador. Usa otra carpeta o apunta a los datos ya extraídos.")
    with zipfile.ZipFile(archive) as zf:
        entries = zf.infolist()
        for item in entries:
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination) or stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError(f"Ruta o enlace no permitido en ZIP: {item.filename}")
        # Un margen conservador evita agotar el disco a mitad de la extracción.
        remaining = sum(max(0, i.file_size - ((destination / i.filename).stat().st_size if (destination / i.filename).is_file() else 0)) for i in entries)
        if shutil.disk_usage(destination).free < remaining + 16 * 1024 * 1024:
            raise OSError("Falta espacio para la extracción; conserva el ZIP y usa otra carpeta")
        json_atomico(pending, identity)
        for index, item in enumerate(entries, 1):
            target = destination / item.filename
            # Sobrescribir los miembros al reanudar corrige archivos incompletos.
            zf.extract(item, destination)
            if index % 500 == 0:
                print(f"Extracción: {index}/{len(entries)}", flush=True)
    json_atomico(marker, identity)
    pending.unlink(missing_ok=True)
    return destination



Writing /content/Tarea2_DL/codigo_parte1/descarga_fjordfish.py


In [ ]:
%%writefile -a /content/Tarea2_DL/codigo_parte1/descarga_fjordfish.py
def preparar_descarga(data_dir, destination=None, archive_path=None, allow_download=False):
    data_dir = Path(data_dir)
    destination = Path(destination) if destination else data_dir / "extracted"
    archive = Path(archive_path) if archive_path else data_dir / ARCHIVE_NAME
    # Un ZIP suministrado también debe coincidir con la versión oficial fijada.
    entry = entrada_oficial() if allow_download else {
        "checksum": "md5:" + PUBLISHED_MD5, "size": None,
        "url": f"https://zenodo.org/records/{RECORD}/files/{ARCHIVE_NAME}?download=1",
        "metadata_source": "published_checksum_manual_zip",
    }
    if not archive.exists() and not allow_download:
        raise FileNotFoundError(f"No existe {archive}. Activa DESCARGAR_DATASET o proporciona ZIP_LOCAL/DATASET_DIR.")
    archive = descargar_reanudable(archive, entry) if allow_download else archive
    verificar_zip(archive, entry)
    root = extraer_seguro(archive, destination, entry["checksum"])
    json_atomico(data_dir / "source.json", {"record": RECORD, "url": f"https://zenodo.org/records/{RECORD}",
        "archive": str(archive.resolve()), "extracted": str(root), "verified": True, **entry})
    return root

def resolver_catalogo(root, output, yaml_override=None):
    import yaml
    root, output = Path(root), Path(output)
    files = [Path(yaml_override)] if yaml_override else sorted(set(root.rglob("data.yaml")) | set(root.rglob("data.yml")))
    if len(files) > 1:
        raise ValueError("Hay varios data.yaml: especifica YAML_CATALOGO sin adivinar las clases")
    if files:
        path = files[0]
        data = yaml.safe_load(path.read_text(encoding="utf-8-sig"))
        raw = data.get("names")
        if isinstance(raw, list):
            raw = dict(enumerate(raw))
        if not isinstance(raw, dict) or not raw:
            raise ValueError("El YAML debe contener names como lista o diccionario")
        names = {int(k): re.sub(r"^\d+\s*:\s*", "", str(v)).strip() for k, v in raw.items()}
        if sorted(names) != list(range(len(names))) or len(set(names.values())) != len(names) or any(not n for n in names.values()):
            raise ValueError("Catálogo con IDs no contiguos, nombres vacíos o duplicados")
        if "nc" in data and int(data["nc"]) != len(names):
            raise ValueError("El campo nc no coincide con el número de nombres del YAML")
        ordered = [names[i] for i in range(len(names))]
        origin = {"source": "dataset_yaml", "path": str(path.resolve()), "sha256": hash_archivo(path)}
    else:
        ordered = list(PUBLISHED_NAMES)
        origin = {"source": "published_table_27_classes", "url": f"https://zenodo.org/records/{RECORD}"}
    output.mkdir(parents=True, exist_ok=True)
    # El catálogo normalizado va fuera de los originales; la auditoría valida todos los IDs.
    normalized = output / "catalogo_validado.yaml"
    normalized.write_text(yaml.safe_dump({"nc": len(ordered), "names": dict(enumerate(ordered))}, sort_keys=False), encoding="utf-8")
    json_atomico(output / "catalogo_origen.json", {**origin, "classes": len(ordered), "names": ordered,
        "differs_from_published_table": ordered != PUBLISHED_NAMES})
    print(f"Catálogo: {len(ordered)} clases; origen: {origin['source']}. Se validarán los IDs de cada etiqueta.")
    return normalized


Appending to /content/Tarea2_DL/codigo_parte1/descarga_fjordfish.py


### 4.3 Auditoría y particiones — `preparacion_parte1.py` (1 bloques)


In [ ]:
%%writefile /content/Tarea2_DL/codigo_parte1/preparacion_parte1.py
"""Preparación común sin duplicar los GB de imágenes al exportar las particiones."""
from pathlib import Path
from collections import Counter
import hashlib
import json
from descarga_fjordfish import resolver_catalogo
from base_parte1 import (
    auditar, depurar, firma_archivos, calcular_sha256, leer_json, guardar_json,
    huella_dataset, asignar_grupos, revisar_particiones, revisar_manifiesto,
    formato_coco, calcular_normalizacion,
)

def particiones_sin_copias(auditoria, output, ratios=(0.7, 0.15, 0.15), seed=42):
    if len(ratios) != 3 or any(x <= 0 for x in ratios) or abs(sum(ratios) - 1) > 1e-8:
        raise ValueError("Se requieren tres proporciones positivas que sumen uno")
    if not auditoria["valid"]:
        raise ValueError("No se pueden crear particiones con errores pendientes")
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    fingerprint = huella_dataset(auditoria)
    path = output / "manifest.json"
    if path.exists():
        manifest = leer_json(path)
        if (manifest["dataset_fingerprint"] != fingerprint or manifest["seed"] != seed or
                manifest["ratios_requested"] != list(ratios)):
            raise ValueError("Las particiones guardadas corresponden a otros datos o parámetros. Usa otra carpeta de preparación.")
        revisar_manifiesto(auditoria, manifest)
        return manifest
    partitions = asignar_grupos(auditoria, ratios, seed)
    revisar_particiones(auditoria, partitions)
    by_id = {r["id"]: r for r in auditoria["images"]}
    statistics = {}
    for name, ids in partitions.items():
        counts = Counter(a["category_id"] for i in ids for a in by_id[i]["annotations"])
        statistics[name] = {"images": len(ids), "groups": len({by_id[i]["group_id"] for i in ids}),
            "objects_by_class": {str(cat["id"]): counts[cat["id"]] for cat in auditoria["categories"]},
            "classes_without_gt": [cat["id"] for cat in auditoria["categories"] if not counts[cat["id"]]]}
        guardar_json(output / f"{name}.ids.json", ids)
        guardar_json(output / f"{name}.coco.json", formato_coco(auditoria, ids))
    manifest = {"schema_version": 1, "seed": seed, "ratios_requested": list(ratios),
        "dataset_fingerprint": fingerprint, "partitions": partitions, "statistics": statistics,
        "warnings": auditoria["warnings"], "export": "ids_and_coco_no_image_copies",
        "method": "original greedy grouped image/class balance; singleton classes prefer train"}
    guardar_json(path, manifest)
    return manifest

def preparar_datos_parte1(root, output, *, size=512, seed=42, ratios=(0.7, 0.15, 0.15),
                          yaml_override=None, groups_csv=None, near_distance=4, curate=True, reuse=True):
    root, output = Path(root).resolve(), Path(output)
    if not root.is_dir():
        raise FileNotFoundError(f"No existe la carpeta de datos extraídos: {root}")
    output.mkdir(parents=True, exist_ok=True)
    yaml_path = resolver_catalogo(root, output / "catalogo", yaml_override)
    identity = {"root": str(root), "signature": firma_archivos(root), "catalog_sha256": calcular_sha256(yaml_path),
                "groups_sha256": calcular_sha256(groups_csv) if groups_csv else None, "near_distance": near_distance}
    cached_path, identity_path = output / "auditoria/audit.json", output / "auditoria/input.json"
    if reuse and cached_path.exists() and identity_path.exists() and leer_json(identity_path) == identity:
        raw = leer_json(cached_path)
        print("Auditoría original reutilizada: entradas coincidentes")
    else:
        raw = auditar(root, output / "auditoria", yaml_path=yaml_path, groups_csv=groups_csv,
                      allow_missing=False, near_distance=near_distance)
        guardar_json(identity_path, identity)
    audit = depurar(raw, output / "curacion") if curate else raw
    if not audit["valid"]:
        raise ValueError(f"Hay errores pendientes: revisa {cached_path}. {audit['errors'][:8]}")
    manifest = particiones_sin_copias(audit, output / "particiones", ratios, seed)
    statistics_path = output / "particiones/train_statistics.json"
    statistics = leer_json(statistics_path) if statistics_path.exists() and reuse else None
    if (not statistics or statistics.get("size") != size or statistics.get("split") != "train" or
            statistics.get("dataset_fingerprint") != manifest["dataset_fingerprint"]):
        statistics = calcular_normalizacion(audit, manifest, size=size)
        guardar_json(statistics_path, statistics)
    guardar_json(output / "dataset_ready.json", {"dataset_fingerprint": manifest["dataset_fingerprint"],
        "root": str(root), "classes": len(audit["categories"]), "images": audit["summary"]["images"],
        "objects": audit["summary"]["objects"], "partitions": manifest["statistics"], "curation": audit.get("curation"),
        "warnings": audit["warnings"]})
    return audit, manifest, statistics


Writing /content/Tarea2_DL/codigo_parte1/preparacion_parte1.py


### 4.4 Entrenamiento, evaluación y gráficos — `ejecucion_parte1.py` (2 bloques)


In [ ]:
%%writefile /content/Tarea2_DL/codigo_parte1/ejecucion_parte1.py
"""Ejecución de la parte 1 sobre los modelos del notebook original.

No contiene conclusiones ni justificaciones de diseño. El entrenamiento usa solo train;
la selección de pesos y umbral usa val; test se procesa después de terminar ambos modelos.
"""
from pathlib import Path
import gc
import math
import pickle
import json
import csv
import hashlib
import importlib.metadata
import time
import random
import copy
import numpy as np
from PIL import Image, ImageDraw
import torch
from base_parte1 import (
    DatosDeteccion, crear_detector, adamw, paso_entrenamiento, fijar_semilla,
    revisar_manifiesto, guardar_checkpoint, recuperar_checkpoint,
    guardar_json, leer_json, calcular_sha256, formato_coco, predicciones_coco,
    evaluar, elegir_umbral,
)

PIPELINE_VERSION = 1
MODELOS = ("cnn_ssd", "yolo_scratch")

def validar_configuracion(config):
    config = copy.deepcopy(config)
    if set(config) != set(MODELOS):
        raise ValueError(f"Configura exactamente {MODELOS}")
    for name, cfg in config.items():
        required = {"epochs", "batch_size", "learning_rate", "weight_decay", "augment"}
        if set(cfg) != required:
            raise ValueError(f"{name}: campos requeridos {sorted(required)}")
        for key in ("epochs", "batch_size"):
            if type(cfg[key]) is not int or cfg[key] < 1:
                raise ValueError(f"{name}: completa {key} con un entero positivo decidido por el grupo")
        if type(cfg["augment"]) is not bool:
            raise ValueError(f"{name}: augment debe ser True o False (reflejo horizontal con probabilidad 0.5)")
        for key in ("learning_rate", "weight_decay"):
            value = cfg[key]
            if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
                raise ValueError(f"{name}: {key} debe ser numérico y finito")
        if cfg["learning_rate"] <= 0 or cfg["weight_decay"] < 0:
            raise ValueError(f"{name}: learning_rate > 0 y weight_decay >= 0")
    return config

def sincronizar(device):
    if str(device).startswith("cuda"):
        torch.cuda.synchronize(device)

def crear_contexto(auditoria, manifiesto, estadisticas, size, seed, device, entorno,
                   selection_metric="map_50_95", eval_batch_size=2, data_scope="fjordfish"):
    revisar_manifiesto(auditoria, manifiesto)
    if type(size) is not int or size < 128 or size % 64:
        raise ValueError("Usa un tamaño múltiplo de 64 y no menor que 128")
    if selection_metric not in {"map_50_95", "ap_50"}:
        raise ValueError("La selección debe usar map_50_95 o ap_50 de validación")
    if type(eval_batch_size) is not int or eval_batch_size < 1:
        raise ValueError("eval_batch_size debe ser positivo")
    if (estadisticas.get("split") != "train" or estadisticas.get("size") != size or
            estadisticas.get("dataset_fingerprint") != manifiesto["dataset_fingerprint"]):
        raise ValueError("Las estadísticas deben corresponder al mismo train y tamaño")
    sources = {}
    for filename in ("base_parte1.py", "ejecucion_parte1.py"):
        path = Path(__file__).with_name(filename)
        sources[filename] = calcular_sha256(path)
    return {"data_scope": data_scope, "pipeline_version": PIPELINE_VERSION, "dataset_fingerprint": manifiesto["dataset_fingerprint"],
            "size": size, "seed": seed, "device": str(device), "selection_metric": selection_metric,
            "eval_batch_size": eval_batch_size, "classes": auditoria["categories"],
            "statistics": estadisticas, "sources_sha256": sources,
            "packages": entorno.get("packages", {}), "hardware": entorno.get("devices", []),
            "gpu_verified": entorno.get("gpu_test_passed", False),
            "normalization": "train_statistics_for_cnn; original_yolo_input_0_1",
            "postprocess": {"score_min": 0.001, "nms_iou": 0.5, "max_detections": 100},
            "threshold_selection": "validation_f1_on_original_0.05_to_0.95_grid"}

def firma(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()

def predecir_particion(detector, auditoria, manifiesto, split, batch_size=2):
    if split not in {"val", "test"}:
        raise ValueError("La evaluación de esta ejecución admite val o test")
    ids = list(manifiesto["partitions"][split])
    by_id = {r["id"]: r for r in auditoria["images"]}
    device = next(detector.parameters()).device
    sincronizar(device)
    started = time.perf_counter()
    predictions = []
    for start in range(0, len(ids), batch_size):
        batch_ids = ids[start:start + batch_size]
        images = []
        for identifier in batch_ids:
            row = by_id[identifier]
            with Image.open(Path(auditoria["root"]) / row["file_name"]) as raw:
                if raw.size != (row["width"], row["height"]):
                    raise ValueError("Una imagen cambió de tamaño desde la auditoría")
                images.append(raw.convert("RGB"))
        output = detector.predict(images)
        if len(output) != len(batch_ids):
            raise RuntimeError("El detector no devolvió una salida por imagen")
        predictions.extend(predicciones_coco(batch_ids, output))
    sincronizar(device)
    seconds = time.perf_counter() - started
    return predictions, {"images": len(ids), "end_to_end_seconds": seconds,
                         "images_per_second": len(ids) / seconds if seconds else None,
                         "scope": "image_loading_preprocessing_model_and_postprocessing",
                         "batch_size": batch_size, "device": str(device)}

def _reanudar(output, detector, optimizer, context, signature):
    candidates = []
    failures = []
    for name in ("last.pt", "best.pt"):
        path = output / name
        if not path.is_file():
            continue
        try:
            state = torch.load(path, map_location="cpu", weights_only=True)
        except (OSError, RuntimeError, EOFError, ValueError, pickle.UnpicklingError) as error:
            failures.append(f"{name}: {error}")
            continue
        metadata = state.get("training", {})
        if metadata.get("signature") != signature:
            raise ValueError(f"{path}: configuración/datos/código incompatibles. Usa otro RUN_ID.")
        if metadata.get("epoch") != len(metadata.get("history", [])):
            raise ValueError(f"Historial incompatible en {path}")
        candidates.append((metadata["epoch"], name == "last.pt", path, metadata))
    if not candidates:
        if failures:
            raise RuntimeError("No hay un checkpoint legible: " + "; ".join(failures))
        return None
    _, _, path, metadata = max(candidates, key=lambda item: item[:2])
    # Una interrupción entre best.pt y last.pt puede dejar best una época más adelante.
    recuperar_checkpoint(path, detector, optimizer, context["dataset_fingerprint"])
    if failures:
        print("Se recuperó un checkpoint válido; incidencias: " + "; ".join(failures))
    print(f"Reanudado {detector.kind}: época {metadata['epoch']} desde {path.name}", flush=True)
    return metadata

def entrenar_modelo(auditoria, manifiesto, context, kind, cfg, output, resume=True):
    revisar_manifiesto(auditoria, manifiesto)
    if kind not in MODELOS:
        raise ValueError("Parte 1: no se admiten modelos preentrenados")
    # Se valida incluso al invocar esta función sin el coordinador.
    validar_configuracion({name: cfg for name in MODELOS})
    if not formato_coco(auditoria, manifiesto["partitions"]["val"])["annotations"]:
        raise ValueError("Validación no tiene objetos: no se puede seleccionar el checkpoint por AP")
    output = Path(output)
    identity = {"kind": kind, "configuration": cfg, "context": context,
                "optimizer": "AdamW, weight_decay solo en matrices de pesos; sin scheduler",
                "augmentation": "hflip_p0.5_train_only" if cfg["augment"] else "none"}
    signature = firma(identity)
    experiment_path = output / "experiment.json"
    if experiment_path.exists():
        if leer_json(experiment_path).get("signature") != signature:
            raise ValueError(f"{output}: otro experimento ocupa esta carpeta. Cambia RUN_ID.")
        if not resume:
            raise ValueError("La carpeta ya contiene un experimento; usa otro RUN_ID para comenzar desde cero")
    elif output.exists() and any(output.iterdir()):
        raise ValueError(f"{output}: carpeta no vacía sin identidad de experimento; usa otra carpeta")
    output.mkdir(parents=True, exist_ok=True)
    guardar_json(experiment_path, {"signature": signature, **identity})
    fijar_semilla(context["seed"])
    detector = crear_detector(kind, len(auditoria["categories"]), context["size"], statistics=context["statistics"]).to(context["device"])
    if detector.provenance.get("pretrained") is not False:
        raise RuntimeError("La parte 1 requiere inicialización sin preentrenamiento")
    optimizer = adamw(detector, cfg["learning_rate"], cfg["weight_decay"])
    guardar_json(output / "parameters.json", detector.parameter_report())
    (output / "architecture.txt").write_text(str(detector.network) + "\n", encoding="utf-8")
    metadata = _reanudar(output, detector, optimizer, context, signature) if resume else None
    if metadata:
        history = metadata["history"]
        best_score, best_epoch = metadata["best_score"], metadata["best_epoch"]
        global_step = metadata["global_step"]
        training_seconds, validation_seconds = metadata["training_seconds"], metadata["validation_seconds"]
    else:
        if (output / "history.json").exists():
            raise RuntimeError("Hay historial sin checkpoint recuperable. Revisa los archivos o usa otro RUN_ID.")
        history, best_score, best_epoch = [], None, None
        global_step, training_seconds, validation_seconds = 0, 0.0, 0.0
    # El Dataset de entrenamiento se construye exclusivamente con los IDs de train.
    data = DatosDeteccion(auditoria, manifiesto["partitions"]["train"], size=context["size"], augment=cfg["augment"])
    validation_gt = formato_coco(auditoria, manifiesto["partitions"]["val"])
    for epoch in range(len(history) + 1, cfg["epochs"] + 1):
        indices = list(range(len(data)))
        random.Random(context["seed"] + epoch).shuffle(indices)
        losses = []
        sincronizar(context["device"])
        started = time.perf_counter()
        for start in range(0, len(indices), cfg["batch_size"]):
            examples = [data[i] for i in indices[start:start + cfg["batch_size"]]]
            images = [image.to(context["device"]) for image, _ in examples]
            targets = [{k: v.to(context["device"]) for k, v in target.items()} for _, target in examples]
            losses.append(paso_entrenamiento(detector, optimizer, images, targets))
            global_step += 1
            if len(losses) % 50 == 0:
                print(f"{kind} época {epoch}: lote {len(losses)}, pérdida={losses[-1]:.5f}", flush=True)
        sincronizar(context["device"])
        train_elapsed = time.perf_counter() - started
        training_seconds += train_elapsed
        validation_started = time.perf_counter()
        predictions, timing = predecir_particion(detector, auditoria, manifiesto, "val", context["eval_batch_size"])
        metrics = evaluar(validation_gt, predictions)
        val_elapsed = time.perf_counter() - validation_started
        validation_seconds += val_elapsed
        score = metrics[context["selection_metric"]]
        if score is None or not math.isfinite(score):
            raise RuntimeError("La métrica de validación no es finita; no se seleccionará un checkpoint")
        improved = best_score is None or score > best_score
        if improved:
            best_score, best_epoch = score, epoch
        history.append({"epoch": epoch, "mean_batch_loss": float(np.mean(losses)), "batches": len(losses),
                        "train_images": len(data), "global_step": global_step,
                        "train_seconds": train_elapsed, "validation_seconds": val_elapsed,
                        "val_map_50_95": metrics["map_50_95"], "val_ap_50": metrics["ap_50"],
                        "validation_inference": timing})
        metadata = {"signature": signature, "epoch": epoch, "history": history,
                    "best_score": best_score, "best_epoch": best_epoch, "global_step": global_step,
                    "training_seconds": training_seconds, "validation_seconds": validation_seconds}
        if improved:
            guardar_checkpoint(output / "best.pt", detector, optimizer, global_step, context["dataset_fingerprint"], training=metadata)
        guardar_checkpoint(output / "last.pt", detector, optimizer, global_step, context["dataset_fingerprint"], training=metadata)
        guardar_json(output / "history.json", history)
        print(f"{kind}: época {epoch}/{cfg['epochs']} | pérdida {history[-1]['mean_batch_loss']:.5f} | val mAP50-95 {metrics['map_50_95']:.4f} | val AP50 {metrics['ap_50']:.4f}", flush=True)
    if not history or not (output / "best.pt").is_file():
        raise RuntimeError("El entrenamiento no produjo un checkpoint seleccionable")
    guardar_json(output / "history.json", history)
    best_state = torch.load(output / "best.pt", map_location="cpu", weights_only=True)
    if best_state["training"]["epoch"] != best_epoch or best_state["training"]["signature"] != signature:
        raise RuntimeError("best.pt no coincide con la selección registrada")
    summary = {"kind": kind, "signature": signature, "configuration": cfg, "epochs_completed": len(history),
               "best_epoch": best_epoch, "best_validation_score": best_score,
               "selection_metric": context["selection_metric"], "training_seconds": training_seconds,
               "validation_seconds": validation_seconds, "global_step": global_step,
               "initialization": "random", "pretrained": False, "best_checkpoint_sha256": calcular_sha256(output / "best.pt"),
               "dataset_fingerprint": context["dataset_fingerprint"], "train_images": len(data)}
    guardar_json(output / "training_summary.json", summary)
    del detector, optimizer, best_state
    gc.collect()
    if str(context["device"]).startswith("cuda"):
        torch.cuda.empty_cache()
    return summary



Writing /content/Tarea2_DL/codigo_parte1/ejecucion_parte1.py


In [ ]:
%%writefile -a /content/Tarea2_DL/codigo_parte1/ejecucion_parte1.py
def evaluar_modelo_final(auditoria, manifiesto, context, kind, output, summary):
    revisar_manifiesto(auditoria, manifiesto)
    output = Path(output)
    checkpoint = output / "best.pt"
    actual_hash = calcular_sha256(checkpoint)
    if actual_hash != summary["best_checkpoint_sha256"]:
        raise ValueError("Los pesos seleccionados cambiaron antes de evaluar")
    evaluation_signature = firma({"experiment": summary["signature"], "checkpoint_sha256": actual_hash,
                                  "dataset_fingerprint": context["dataset_fingerprint"], "evaluation": context})
    result_path = output / "final_evaluation.json"
    if result_path.exists():
        result = leer_json(result_path)
        if result.get("signature") != evaluation_signature:
            raise ValueError("La evaluación guardada corresponde a otros pesos o configuración")
        for name in ("validation_predictions.json", "threshold.json", "test_predictions.json"):
            if not (output / name).is_file():
                raise RuntimeError(f"Falta {name} de la evaluación guardada. Revisa el respaldo antes de repetir test.")
        print(f"Evaluación final guardada reutilizada: {kind}")
        return result
    fijar_semilla(context["seed"])
    detector = crear_detector(kind, len(auditoria["categories"]), context["size"], statistics=context["statistics"]).to(context["device"])
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    if state.get("training", {}).get("signature") != summary["signature"]:
        raise ValueError("Checkpoint incompatible con el experimento seleccionado")
    detector.load_state_dict(state["model"])
    del state
    validation_predictions, validation_timing = predecir_particion(detector, auditoria, manifiesto, "val", context["eval_batch_size"])
    validation_gt = formato_coco(auditoria, manifiesto["partitions"]["val"])
    threshold = elegir_umbral(validation_gt, validation_predictions, "val")
    selected = threshold["selected"]["threshold"]
    guardar_json(output / "validation_predictions.json", validation_predictions)
    guardar_json(output / "threshold.json", threshold)
    validation_metrics = evaluar(validation_gt, validation_predictions, selected)
    # Este es el único punto del flujo que hace inferencia sobre test.
    test_predictions, test_timing = predecir_particion(detector, auditoria, manifiesto, "test", context["eval_batch_size"])
    test_metrics = evaluar(formato_coco(auditoria, manifiesto["partitions"]["test"]), test_predictions, selected)
    guardar_json(output / "test_predictions.json", test_predictions)
    result = {"signature": evaluation_signature, "kind": kind, "checkpoint_sha256": actual_hash,
              "best_epoch": summary["best_epoch"], "threshold_source": "val", "threshold": selected,
              "validation": validation_metrics, "test": test_metrics,
              "validation_inference_timing": validation_timing, "test_inference_timing": test_timing,
              "context": context, "training": summary}
    guardar_json(result_path, result)
    del detector
    gc.collect()
    if str(context["device"]).startswith("cuda"):
        torch.cuda.empty_cache()
    return result

def exportar_comparacion(results, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    rows, per_class = [], []
    for kind in MODELOS:
        result = results[kind]
        metrics, training = result["test"], result["training"]
        point = metrics["operating_point"]
        context = result["context"]
        rows.append({"model": kind, "initialization": "random", "pretrained": False,
                     "seed": context["seed"], "epochs": training["epochs_completed"],
                     "best_epoch": result["best_epoch"], "train_images": training["train_images"],
                     "validation_images": result["validation"]["images"], "test_images": metrics["images"],
                     "map_50_95": metrics["map_50_95"], "ap_50": metrics["ap_50"],
                     "precision": point["precision"], "recall": point["recall"], "f1": point["f1"],
                     "threshold_from_val": result["threshold"], "tp": point["tp"], "fp": point["fp"], "fn": point["fn"],
                     "training_seconds": training["training_seconds"], "validation_seconds": training["validation_seconds"],
                     "test_end_to_end_seconds": result["test_inference_timing"]["end_to_end_seconds"],
                     "test_images_per_second": result["test_inference_timing"]["images_per_second"],
                     "device": context["device"], "gpu_verified": context["gpu_verified"]})
        per_class.extend({"model": kind, **cat} for cat in metrics["per_class"])
    for filename, data in (("comparacion_test.csv", rows), ("ap_por_clase_test.csv", per_class)):
        with (output / filename).open("w", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(data[0]))
            writer.writeheader()
            writer.writerows(data)
    guardar_json(output / "comparacion_test.json", rows)
    return rows

def graficar_experimento(results, output):
    import matplotlib.pyplot as plt
    output = Path(output)
    for kind in MODELOS:
        history = leer_json(output / kind / "history.json")
        epochs = [h["epoch"] for h in history]
        fig, axes = plt.subplots(1, 3, figsize=(14, 3.7))
        axes[0].plot(epochs, [h["mean_batch_loss"] for h in history])
        axes[0].set(title="Pérdida media por lote (train)", xlabel="Época", ylabel="Pérdida")
        axes[1].plot(epochs, [h["val_map_50_95"] for h in history], label="mAP50–95")
        axes[1].plot(epochs, [h["val_ap_50"] for h in history], label="AP50")
        axes[1].set(title="Validación", xlabel="Época", ylabel="AP", ylim=(0, 1))
        axes[1].legend()
        axes[2].plot(epochs, [h["train_seconds"] for h in history])
        axes[2].set(title="Tiempo de entrenamiento por época", xlabel="Época", ylabel="Segundos")
        fig.suptitle(kind)
        fig.tight_layout()
        fig.savefig(output / f"curvas_{kind}.png", dpi=150)
        plt.close(fig)
    keys = ("map_50_95", "ap_50", "precision", "recall", "f1")
    rows = leer_json(output / "comparacion_test.json")
    fig, ax = plt.subplots(figsize=(9, 4))
    positions = np.arange(len(keys))
    for i, row in enumerate(rows):
        ax.bar(positions + (i - 0.5) * 0.36, [row[k] if row[k] is not None else np.nan for k in keys], width=0.36, label=row["model"])
    ax.set_xticks(positions, keys)
    ax.set(ylim=(0, 1), ylabel="Valor", title="Test: métricas con pesos seleccionados en validación")
    ax.legend()
    fig.tight_layout()
    fig.savefig(output / "comparacion_test.png", dpi=150)
    plt.close(fig)

def mostrar_detecciones_finales(auditoria, manifiesto, results, output, count=6):
    output = Path(output)
    by_id = {r["id"]: r for r in auditoria["images"]}
    names = {c["id"]: c["name"] for c in auditoria["categories"]}
    for kind in MODELOS:
        predictions = leer_json(output / kind / "test_predictions.json")
        folder = output / kind / "muestra_test"
        folder.mkdir(exist_ok=True)
        for identifier in manifiesto["partitions"]["test"][:count]:
            row = by_id[identifier]
            with Image.open(Path(auditoria["root"]) / row["file_name"]) as raw:
                image = raw.convert("RGB")
            draw = ImageDraw.Draw(image)
            for ann in row["annotations"]:
                draw.rectangle(ann["bbox_xyxy"], outline="lime", width=2)
                draw.text(tuple(ann["bbox_xyxy"][:2]), "GT: " + names[ann["category_id"]], fill="lime")
            for pred in predictions:
                if pred["image_id"] != identifier or pred["score"] < results[kind]["threshold"]:
                    continue
                x, y, w, h = pred["bbox"]
                draw.rectangle([x, y, x + w, y + h], outline="red", width=2)
                draw.text((x, y + 12), f"Pred: {names[pred['category_id']]} {pred['score']:.2f}", fill="red")
            image.thumbnail((1280, 1280))
            image.save(folder / f"{identifier:06d}.jpg")

def ejecutar_parte1(auditoria, manifiesto, estadisticas, config, output, *, size=512, seed=42,
                    device="cuda", entorno=None, selection_metric="map_50_95", eval_batch_size=2,
                    resume=True, plots=True, samples=True, data_scope="fjordfish"):
    config = validar_configuracion(config)
    context = crear_contexto(auditoria, manifiesto, estadisticas, size, seed, device, entorno or {},
                            selection_metric, eval_batch_size, data_scope)
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    summaries = {}
    # Ambos entrenamientos terminan antes de comenzar la evaluación de test.
    for kind in MODELOS:
        summaries[kind] = entrenar_modelo(auditoria, manifiesto, context, kind, config[kind], output / kind, resume)
    results = {kind: evaluar_modelo_final(auditoria, manifiesto, context, kind, output / kind, summaries[kind]) for kind in MODELOS}
    rows = exportar_comparacion(results, output)
    if plots:
        graficar_experimento(results, output)
    if samples:
        mostrar_detecciones_finales(auditoria, manifiesto, results, output)
    guardar_json(output / "estado_parte1.json", {"scope": context.get("data_scope", "fjordfish"), "stage": "full_train_val_test", "completed": True,
        "dataset_fingerprint": context["dataset_fingerprint"], "models": list(MODELOS), "context": context,
        "outputs": ["comparacion_test.csv", "ap_por_clase_test.csv", "comparacion_test.json"],
        "analysis": "Pendiente de elaboración por el grupo"})
    return {"results": results, "comparison": rows}


Appending to /content/Tarea2_DL/codigo_parte1/ejecucion_parte1.py


### 4.5 Importar módulos y guardar una copia de respaldo

El notebook conserva el código en Google Drive para reproducir los resultados de cada ejecución. Si modificas código o hiperparámetros después, cambia `RUN_ID` antes de iniciar otro experimento.


In [ ]:
# Guardar copia verificable de los archivos usados en esta ejecución.
import hashlib
for archivo in ('base_parte1.py','descarga_fjordfish.py','preparacion_parte1.py','ejecucion_parte1.py'):
    origen=CODIGO / archivo
    destino=RESULTADOS / 'codigo' / archivo
    destino.parent.mkdir(parents=True, exist_ok=True)
    codigo=origen.read_text(encoding='utf-8')
    if destino.exists() and destino.read_text(encoding='utf-8') != codigo:
        raise RuntimeError(f'El código cambió dentro de {RUN_ID}. Cambia RUN_ID para ejecutar otro experimento.')
    destino.write_text(codigo,encoding='utf-8')
if str(CODIGO) not in sys.path:
    sys.path.insert(0,str(CODIGO))
print('Código fuente preparado y respaldado')


✅ Código fuente preparado y respaldado


In [ ]:
from base_parte1 import fijar_semilla, revisar_entorno, probar_modelos, crear_datos_prueba, guardar_json, leer_json
from descarga_fjordfish import preparar_descarga
from preparacion_parte1 import preparar_datos_parte1
from ejecucion_parte1 import ejecutar_parte1, validar_configuracion

fijar_semilla(SEED)
entorno = revisar_entorno(RESULTADOS / "entorno.json", require_gpu=EXIGIR_GPU)
DEVICE = "cuda" if entorno["gpu_test_passed"] else "cpu"
print(json.dumps(entorno, indent=2, ensure_ascii=False))


{
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "packages": {
    "torch": "2.11.0+cu130",
    "torchvision": "0.26.0+cu130",
    "ultralytics": "8.4.174",
    "numpy": "2.1.3",
    "Pillow": "11.3.0",
    "PyYAML": "6.0.3",
    "pycocotools": "2.0.11",
    "pytest": "8.4.2"
  },
  "cuda_available": true,
  "gpu_test_passed": true,
  "status": "gpu_verified",
  "devices": [
    {
      "name": "Tesla T4",
      "memory_bytes": 15637086208
    }
  ],
  "torchvision_nms_cpu": true,
  "cuda_runtime": "13.0",
  "timestamp_unix": 1793498357.5892057
}


## 5. Integración con datos sintéticos

La prueba realiza dos pasos por modelo y comprueba actualización de parámetros, pérdidas finitas y restauración de checkpoints. No usa ni descarga FjordFish.


In [ ]:
if PRUEBA_INTEGRACION:
    reporte_integracion = probar_modelos(
        RESULTADOS / "integracion_sintetica.json", kinds=("cnn_ssd", "yolo_scratch"),
        device=DEVICE, size=128, steps=2, num_classes=2,
    )
    print({k: v["passed"] for k, v in reporte_integracion["models"].items()})
    if not reporte_integracion["passed"]:
        raise RuntimeError("Revisa integracion_sintetica.json antes de continuar")


Prueba breve: cnn_ssd
Prueba breve: yolo_scratch
{'cnn_ssd': True, 'yolo_scratch': True}


## 6. Descarga y preparación de FjordFish

Con `DESCARGAR_DATASET = True` (configuración inicial), este paso consulta la versión 1.1 en Zenodo y descarga **FjordFish.zip (6,1 GB)** directamente al Drive montado en `Mi unidad/Tarea2_DL/data/`.

- Si `FjordFish.zip` ya existe en ese lugar, **lo reutiliza después de comprobar su MD5** contra el registro oficial.
- Si se corta la descarga, deja `FjordFish.zip.part` para intentar reanudarla en la próxima ejecución.
- La extracción se realiza **en el disco temporal de Colab** para evitar duplicar en Drive todos los archivos del dataset.
- Si ya tienes los datos extraídos, utiliza `DATASET_EXISTENTE` en la configuración y se omitirá la descarga.
- Si prefieres aportar un ZIP existente en otra ruta, indica `ZIP_LOCAL` y pon `DESCARGAR_DATASET = False`.

**Aviso:** no se puede asegurar la descarga ni el entrenamiento hasta ejecutarlo realmente en Colab. El tamaño y la integridad del ZIP se comprueban antes de la extracción.


In [ ]:
if MODO_SINTETICO:
    DATOS_ACTUALES = crear_datos_prueba(PROYECTO / "figuras_sinteticas", count=20, seed=123)
    CONFIG_ACTUAL = {
        name: {"epochs": 1, "batch_size": 2, "learning_rate": 0.001, "weight_decay": 0.0001, "augment": False}
        for name in ("cnn_ssd", "yolo_scratch")
    }
    TAMANO_ACTUAL = 128
    print("DIAGNÓSTICO SINTÉTICO: no son resultados de FjordFish")
else:
    CONFIG_ACTUAL = validar_configuracion(CONFIG_MODELOS)
    TAMANO_ACTUAL = TAMANO
    if DATASET_EXISTENTE:
        DATOS_ACTUALES = DATASET_DIR
        if not DATOS_ACTUALES.is_dir():
            raise FileNotFoundError(f"No existe DATASET_EXISTENTE: {DATOS_ACTUALES}")
    else:
        if ZIP_LOCAL is None and not DESCARGAR_DATASET:
            posibles_zip = [DATA_DIR / 'FjordFish.zip', Path('/content/FjordFish.zip'),
                            PROYECTO / 'FjordFish.zip',
                            Path('/content/drive/MyDrive/FjordFish.zip')]
            ZIP_LOCAL = next((str(p) for p in posibles_zip if p.is_file()), None)
            if ZIP_LOCAL is None:
                raise FileNotFoundError(
                    "Falta FjordFish.zip. Súbelo a Google Drive > Mi unidad > Tarea2_DL > data > FjordFish.zip, "
                    "o a /content/FjordFish.zip usando Archivos de Colab. Luego vuelve a ejecutar."
                )
            print("ZIP FjordFish encontrado:", ZIP_LOCAL)
        if DESCARGAR_DATASET and ZIP_LOCAL is None:
            print("📥 Descarga directa: Zenodo → Google Drive")
            print("Destino del ZIP:", DATA_DIR / "FjordFish.zip")
            print("Se verificará el MD5 oficial antes de extraer; una descarga parcial queda en .part")
        DATOS_ACTUALES = preparar_descarga(DATA_DIR, destination=DATASET_DIR, archive_path=ZIP_LOCAL,
                                         allow_download=DESCARGAR_DATASET)
        origen_json = DATA_DIR / "source.json"
        if origen_json.is_file():
            fuente = leer_json(origen_json)
            print("✅ ZIP completo e íntegro; origen:", fuente.get("metadata_source", "Zenodo"))
print("Datos para esta ejecución:", DATOS_ACTUALES)


📥 Descarga directa: Zenodo → Google Drive
Destino del ZIP: /content/drive/MyDrive/Tarea2_DL/data/FjordFish.zip
Se verificará el MD5 oficial antes de extraer; una descarga parcial queda en .part
Descarga: 2.2%
Descarga: 4.4%
Descarga: 6.6%
Descarga: 8.9%
Descarga: 11.1%
Descarga: 13.3%
Descarga: 15.5%
Descarga: 17.7%
Descarga: 19.9%
Descarga: 22.2%
Descarga: 24.4%
Descarga: 26.6%
Descarga: 28.8%
Descarga: 31.0%
Descarga: 33.2%
Descarga: 35.4%
Descarga: 37.7%
Descarga: 39.9%
Descarga: 42.1%
Descarga: 44.3%
Descarga: 46.5%
Descarga: 48.7%
Descarga: 51.0%
Descarga: 53.2%
Descarga: 55.4%
Descarga: 57.6%
Descarga: 59.8%
Descarga: 62.0%
Descarga: 64.2%
Descarga: 66.5%
Descarga: 68.7%
Descarga: 70.9%
Descarga: 73.1%
Descarga: 75.3%
Descarga: 77.5%
Descarga: 79.7%
Descarga: 82.0%
Descarga: 84.2%
Descarga: 86.4%
Descarga: 88.6%
Descarga: 90.8%
Descarga: 93.0%
Descarga: 95.3%
Descarga: 97.5%
Descarga: 99.7%
Extracción: 500/6015
Extracción: 1000/6015
Extracción: 1500/6015
Extracción: 2000/6015
Ext

## 7. Auditoría y particiones comunes

Se conserva la política de depuración del cuaderno original. Los informes registran reparaciones y exclusiones sin modificar los originales. Los errores no contemplados requieren revisión.

Se exportan IDs y anotaciones COCO. Los dos modelos utilizan las mismas particiones y las cajas depuradas del cargador común; no se duplican los GB de imágenes para entrenar YOLO. La normalización de la CNN se calcula exclusivamente con train.


In [ ]:
reporte_auditoria, manifiesto, estadisticas_train = preparar_datos_parte1(
    DATOS_ACTUALES, RESULTADOS / "preparacion", size=TAMANO_ACTUAL, seed=SEED, ratios=PROPORCIONES,
    yaml_override=None if MODO_SINTETICO else YAML_CATALOGO,
    groups_csv=None if MODO_SINTETICO else GRUPOS_CSV,
    near_distance=-1 if MODO_SINTETICO else DISTANCIA_DHASH,
    curate=APLICAR_CURACION, reuse=REUTILIZAR_AUDITORIA,
)
print("Clases:", len(reporte_auditoria["categories"]))
print("Imágenes:", reporte_auditoria["summary"]["images"], "| objetos:", reporte_auditoria["summary"]["objects"])
print("Particiones:", {name: len(ids) for name, ids in manifiesto["partitions"].items()})
print("Advertencias:", reporte_auditoria["warnings"])


Catálogo: 28 clases; origen: dataset_yaml. Se validarán los IDs de cada etiqueta.
Clases: 28
Imágenes: 2930 | objetos: 6236
Particiones: {'train': 2022, 'val': 454, 'test': 454}
Advertencias: ['Sin metadatos de video/secuencia: dHash no garantiza independencia entre escenas', 'Curación explícita: 1 sidecars reasociados y 72 imágenes en cuarentena; originales intactos']


## 8. Entrenamiento completo, validación y test

Cada época recorre todo train. Se guardan last.pt, best.pt, historial, configuración, arquitectura y parámetros. Los mejores pesos se eligen con la métrica configurada de val; un empate conserva el primero.

Después de terminar ambos entrenamientos, se elige el umbral de precisión/recall/F1 mediante F1 en val. Test se evalúa con pesos y umbral ya fijados. Repetir la misma ejecución reutiliza la evaluación final guardada. Cambiar configuración o código requiere otro RUN_ID.

El tiempo de entrenamiento se registra separado de validación. La inferencia final incluye lectura, preparación, modelo y filtrado; no mide aisladamente el kernel GPU. Las pérdidas se grafican por modelo porque tienen definiciones distintas.


In [ ]:
SALIDA_EXPERIMENTOS = RESULTADOS / "experimentos"
informe_parte1 = ejecutar_parte1(
    reporte_auditoria, manifiesto, estadisticas_train, CONFIG_ACTUAL, SALIDA_EXPERIMENTOS,
    size=TAMANO_ACTUAL, seed=SEED, device=DEVICE, entorno=entorno,
    selection_metric=METRICA_SELECCION, eval_batch_size=BATCH_EVALUACION,
    resume=REANUDAR, plots=True, samples=True,
    data_scope="synthetic_diagnostic_only" if MODO_SINTETICO else "fjordfish_v1.1",
)


cnn_ssd época 1: lote 50, pérdida=9.82752
cnn_ssd época 1: lote 100, pérdida=12.39167
cnn_ssd época 1: lote 150, pérdida=7.44161
cnn_ssd época 1: lote 200, pérdida=8.78924
cnn_ssd época 1: lote 250, pérdida=6.71916
cnn_ssd época 1: lote 300, pérdida=5.95664
cnn_ssd época 1: lote 350, pérdida=7.03616
cnn_ssd época 1: lote 400, pérdida=7.26395
cnn_ssd época 1: lote 450, pérdida=6.78679
cnn_ssd época 1: lote 500, pérdida=8.20740
cnn_ssd época 1: lote 550, pérdida=6.40948
cnn_ssd época 1: lote 600, pérdida=6.87755
cnn_ssd época 1: lote 650, pérdida=7.32488
cnn_ssd época 1: lote 700, pérdida=10.40376
cnn_ssd época 1: lote 750, pérdida=7.13053
cnn_ssd época 1: lote 800, pérdida=9.29658
cnn_ssd época 1: lote 850, pérdida=7.82413
cnn_ssd época 1: lote 900, pérdida=5.70426
cnn_ssd época 1: lote 950, pérdida=6.76615
cnn_ssd época 1: lote 1000, pérdida=7.04633
cnn_ssd: época 1/10 | pérdida 8.00614 | val mAP50-95 0.0000 | val AP50 0.0001
cnn_ssd época 2: lote 50, pérdida=7.60136
cnn_ssd época 2: l

## 9. Resultados y archivos generados

Se muestran únicamente los valores calculados y los gráficos guardados. Las justificaciones y el análisis deben elaborarlos los integrantes del grupo.


In [ ]:
from IPython.display import display, Markdown, Image as ImagenNotebook

def mostrar_numero(value):
    return "N/D" if value is None else f"{value:.4f}"

lineas = ["| Modelo | mAP50–95 | AP50 | Precisión | Recall | F1 | Mejor época | Tiempo train (s) |",
          "|---|---:|---:|---:|---:|---:|---:|---:|"]
for fila in informe_parte1["comparison"]:
    valores = [mostrar_numero(fila[k]) for k in ("map_50_95", "ap_50", "precision", "recall", "f1")]
    lineas.append("| " + " | ".join([fila["model"], *valores, str(fila["best_epoch"]), f"{fila['training_seconds']:.1f}"]) + " |")
if MODO_SINTETICO:
    display(Markdown("**DIAGNÓSTICO SINTÉTICO: esta tabla no evalúa FjordFish.**"))
display(Markdown("\n".join(lineas)))
for archivo in ("curvas_cnn_ssd.png", "curvas_yolo_scratch.png", "comparacion_test.png"):
    display(ImagenNotebook(filename=str(SALIDA_EXPERIMENTOS / archivo)))
print("CSV comparativo:", SALIDA_EXPERIMENTOS / "comparacion_test.csv")
print("AP y soporte por clase:", SALIDA_EXPERIMENTOS / "ap_por_clase_test.csv")
print("Muestras con GT verde y predicción roja: experimentos/<modelo>/muestra_test/")
print("Resultados:", RESULTADOS)


## Qué falta antes de una entrega académica

- **Resultados reales**: el notebook calcula métricas reales solo cuando consigue terminar la descarga/extracción, los entrenamientos y el test.
- **Decisiones y justificaciones del grupo**: arquitectura, regularización, épocas, tasa de aprendizaje, aumento de datos y demás criterios. Los valores provisionales no equivalen a una justificación.
- **Presentación**: preparar gráficas, explicación y video con las conclusiones realizadas por el equipo.
- **Uso de IA**: la rúbrica prohíbe usar IA para diseñar soluciones o analizar resultados; requiere declarar el uso y contempla nota mínima por infringirlo. Confirma con el docente que la asistencia técnica recibida es admisible.
- **Colab**: las sesiones pueden desconectarse. Se guarda el entrenamiento por épocas completas en Drive y se reanuda al ejecutar de nuevo con la misma configuración.

- **Descarga desde Zenodo:** se realiza automáticamente si `DESCARGAR_DATASET = True`; requiere suficiente cuota en Drive, espacio de extracción en Colab y una conexión estable. No se incluye el archivo de 6,1 GB dentro del notebook.
